# Faster R-CNN · COCO 사전학습 실험

**범위:** 특징 추출 FE · 부분 미세조정 PT · 전체 미세조정 FT × 학습률 3개.
탐색 9회(seed 0) → 상위 설정 2개의 seed 1·2 추가(4회), 총 13회.
과거 베이스라인 결과는 읽지 않는다. 각 학습은 COCO 가중치에서 독립적으로 시작한다.

**실행:** 1~2번 준비·스모크 → 3번에서 실행할 후보 선택 → 4번 분석 → 5번 제출 파일 생성.
`RUN_CANDIDATES`, `RUN_REPEATS`는 기본 False다. 학습할 때 해당 셀에서 True로 바꾸고 실행한다.
실행 전체를 반복하기보다 필요한 셀을 선택한다. 분석·제출은 재학습 없이 실행할 수 있다.

**환경:** Colab에서는 GPU 런타임을 선택한 뒤 **환경 A → 환경 B**를 실행한다.
패키지가 바뀌었다면 **현재 VM의 Python 세션/커널만 재시작**하고 환경 A → B → 셀 1부터 다시 실행한다.
`uv.lock`을 현재 Colab 커널에 적용하고 Python·OS별 패키지 버전·CUDA·torchvision GPU NMS를 검증한다.
환경 기준은 저장소의 `.python-version`, `pyproject.toml`, `uv.lock`이며 잠금 파일은 자동 갱신하지 않는다.
로컬에서는 `uv sync --locked` 후 해당 `.venv` 커널을 사용한다.

**내장 함수:** 데이터 처리·학습·평가·CSV 생성 코드는 아래 함수 셀에 모두 들어 있다.
이 노트북은 프로젝트의 `detection_data.py`, `faster_experiments.py`, `detection_analysis.py`,
`utils.py`를 import하지 않는다. 공용 코드 파일을 GitHub에 올리거나 Colab에 따로 업로드할 필요가 없다.
GitHub는 기존 환경 기준 파일과 팀의 라벨 수정표를 가져오는 데 사용한다.
학습이 끝나면 셀 17-1에서 비교용 validation 자료를 저장한다.

[torchvision 모델](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.detection.fasterrcnn_resnet50_fpn.html) ·
[TorchMetrics 평가](https://lightning.ai/docs/torchmetrics/stable/detection/mean_average_precision.html)

## 1. 실험 설정·실행 준비

### 환경 A · GitHub 저장소·Python 기준 확인

Colab은 **GPU 런타임**을 먼저 선택한다. 아래 URL·브랜치의 환경 파일을 기준으로 실행한다.
저장소가 없으면 복제하고, 이미 있으면 현재 코드와 커밋을 사용한다.
Python 버전이 다르면 프로젝트 기준 Python 런타임을 선택해야 한다.


In [ ]:
import sys
print(sys.version)

In [ ]:
import sys
import os
import json
import hashlib
import importlib
import importlib.metadata as metadata
import shutil
import subprocess
from pathlib import Path

if sys.version_info < (3, 11):
    raise RuntimeError("현재 Python이 너무 낮습니다. Colab에서 프로젝트 기준 Python 3.12 런타임을 선택하세요.")
import tomllib

REPO_URL = "https://github.com/codeit-ai-15th/pill-object-detection.git"
REPO_BRANCH = "exp/23-faster-rcnn-baseline"
# 별도로 업로드한 저장소를 쓰면 해당 루트의 절대 경로를 입력한다.
REPO_ROOT_OVERRIDE = None
IS_COLAB = sys.platform == "linux" and Path("/content").is_dir()

if REPO_ROOT_OVERRIDE:
    REPO_ROOT = Path(REPO_ROOT_OVERRIDE).expanduser().resolve()
elif IS_COLAB:
    REPO_ROOT = Path("/content/pill-object-detection")
else:
    REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                      if (p / "pyproject.toml").is_file()
                      and (p / "src/pill_detection").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError("저장소를 찾지 못했습니다. REPO_ROOT_OVERRIDE에 저장소 루트를 입력하세요.")

# Colab에 저장소가 없을 때만 복제한다. 기존 체크아웃은 자동 갱신하지 않는다.
if IS_COLAB and not REPO_ROOT.exists():
    result = subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)],
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"}, capture_output=True, text=True,
    )
    if result.returncode:
        raise RuntimeError(f"저장소 복제에 실패했습니다. URL·브랜치·접근 권한을 확인하세요.\n{result.stderr[-3000:]}")

reference_names = (".python-version", "pyproject.toml", "uv.lock")
missing = [name for name in reference_names if not (REPO_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"환경 기준 파일이 없습니다: {missing}. 저장소 전체를 준비하고 환경 A부터 다시 실행하세요.")
branch_result = subprocess.run(["git", "branch", "--show-current"], cwd=REPO_ROOT, capture_output=True, text=True)
if branch_result.returncode:
    raise RuntimeError("Git 저장소를 확인할 수 없습니다. 위 URL의 저장소를 복제한 경로를 지정하세요.")
if IS_COLAB and branch_result.stdout.strip() != REPO_BRANCH:
    raise RuntimeError(f"현재 브랜치가 {REPO_BRANCH}와 다릅니다. 작업을 보관한 뒤 지정 브랜치를 준비하세요.")
commit_result = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, capture_output=True, text=True, check=True)
ENV_REPOSITORY_COMMIT = commit_result.stdout.strip()
ENV_REFERENCE_SHA256 = {name: hashlib.sha256((REPO_ROOT / name).read_bytes()).hexdigest()
                        for name in reference_names}
python_pin = (REPO_ROOT / ".python-version").read_text().strip()
pin_parts = tuple(int(part) for part in python_pin.split("."))
if tuple(sys.version_info[:len(pin_parts)]) != pin_parts:
    raise RuntimeError(f"프로젝트 기준은 Python {python_pin}, 현재는 {sys.version.split()[0]}입니다. "
                       "Colab에서 해당 Python 런타임을 선택하고 다시 실행하세요. 이 셀은 실행 중인 Python을 교체하지 않습니다.")
if IS_COLAB and Path(sys.prefix).resolve() == (REPO_ROOT / ".venv").resolve():
    raise RuntimeError("Colab 기본 Python 커널을 선택하세요. 학습은 별도 .venv가 아닌 현재 Colab 커널에서 실행합니다.")

print("저장소:", REPO_ROOT)
print("브랜치:", branch_result.stdout.strip(), "| 커밋:", ENV_REPOSITORY_COMMIT[:12])
print("Python:", sys.version.split()[0], "| 현재 커널:", sys.executable)
print("uv.lock SHA256:", ENV_REFERENCE_SHA256["uv.lock"])
print("다음: 환경 B에서 잠금 버전을 적용합니다.")


### 환경 B · uv.lock 설치·커널 재시작 안내

Colab의 **현재 커널**에 `uv sync --locked --no-dev --inexact`를 적용한다.
잠금 버전의 프로젝트 패키지를 설치하며, 잠금 목록 외 Colab 패키지는 유지한다.
설치 변경이 있으면 **현재 VM의 Python 세션/커널만 재시작**하고 환경 A → B부터 다시 실행한다.
재시작 안내가 없어졌을 때 셀 1로 넘어간다. 새 런타임을 삭제·생성하면 설치를 다시 해야 한다.

[uv 잠금·동기화](https://docs.astral.sh/uv/concepts/projects/sync/) ·
[uv 설치 대상 환경](https://docs.astral.sh/uv/concepts/projects/config/#project-environment-path)


In [ ]:
if "ENV_REFERENCE_SHA256" not in globals():
    raise RuntimeError("환경 A를 먼저 실행하세요.")


def installed_versions():
    """현재 커널 환경의 설치 패키지 목록을 반환한다.

    Returns:
        list: 패키지 이름과 버전의 정렬된 목록.
    """
    return sorted((dist.metadata["Name"].lower().replace("_", "-"), dist.version)
                  for dist in metadata.distributions() if dist.metadata.get("Name"))


if not IS_COLAB:
    print("로컬 환경은 자동 변경하지 않습니다. 저장소에서 uv sync --locked 후 해당 .venv 커널을 선택하세요.")
else:
    UV_BIN = shutil.which("uv")
    local_uv = Path.home() / ".local/bin/uv"
    if UV_BIN is None:
        if not local_uv.is_file():
            print("프로젝트 README의 공식 설치 방법으로 uv를 준비합니다.")
            result = subprocess.run(
                ["bash", "-o", "pipefail", "-c", "curl -LsSf https://astral.sh/uv/install.sh | sh"],
                capture_output=True, text=True,
            )
            if result.returncode:
                raise RuntimeError(f"uv 설치에 실패했습니다. 네트워크를 확인하고 이 셀을 다시 실행하세요.\n{result.stderr[-3000:]}")
        UV_BIN = str(local_uv)
    packages_before = installed_versions()
    restart_was_required = globals().get("ENV_RESTART_REQUIRED", False)
    # 설치 도중 실패하더라도 바뀐 패키지를 기존 프로세스에서 사용하지 않는다.
    ENV_RESTART_REQUIRED = True
    ENV_SYNC_COMPLETED = False
    print("uv.lock의 버전을 현재 Colab 커널에 설치합니다. 최초 CUDA 패키지 다운로드에는 시간이 걸릴 수 있습니다.")
    result = subprocess.run(
        [UV_BIN, "sync", "--locked", "--no-dev", "--inexact", "--python", sys.executable],
        cwd=REPO_ROOT, env={**os.environ, "UV_PROJECT_ENVIRONMENT": sys.prefix},
        capture_output=True, text=True,
    )
    if result.returncode:
        raise RuntimeError("잠금 환경 설치에 실패했습니다. 아래 원인을 해결한 뒤 커널을 재시작하고 환경 A부터 실행하세요. "
                           f"잠금 파일은 자동 갱신하지 않습니다.\n{result.stderr[-5000:]}")
    if any(hashlib.sha256((REPO_ROOT / name).read_bytes()).hexdigest() != digest
           for name, digest in ENV_REFERENCE_SHA256.items()):
        raise RuntimeError("설치 중 환경 기준 파일이 바뀌었습니다. 원인을 확인하고 커널 재시작 후 환경 A부터 실행하세요.")
    importlib.invalidate_caches()
    ENV_SYNC_COMPLETED = True
    ENV_RESTART_REQUIRED = restart_was_required or packages_before != installed_versions()
    if ENV_RESTART_REQUIRED:
        print("설치 완료. 다음 셀을 실행하기 전에 Colab 메뉴에서 세션/커널을 재시작하세요.")
        print("런타임 삭제·연결 해제 대신 현재 VM의 Python 세션만 재시작하고, 환경 A → B → 셀 1 순서로 다시 실행하세요.")
    else:
        print("현재 커널의 패키지가 잠금 환경과 같습니다. 셀 1에서 버전·CUDA를 확인하세요.")


### 셀 1 · 잠금 버전·CUDA 검증

환경 B에 재시작 안내가 나왔다면 Python 세션만 재시작하고 환경 A → B부터 다시 실행한다.
검증 다음에는 바로 아래 내장 함수 A~D를 순서대로 실행하고 셀 2로 넘어간다.


In [ ]:
if "ENV_REFERENCE_SHA256" not in globals():
    raise RuntimeError("환경 A → 환경 B를 먼저 실행하세요.")
if globals().get("ENV_RESTART_REQUIRED", False):
    raise RuntimeError("패키지 설치 후 커널 재시작이 필요합니다. 현재 VM의 Python 세션만 재시작하고 환경 A → B → 셀 1을 실행하세요.")
if IS_COLAB and not globals().get("ENV_SYNC_COMPLETED", False):
    raise RuntimeError("환경 B의 설치가 완료되지 않았습니다. 환경 B를 먼저 실행하세요.")

from packaging.markers import Marker
from packaging.requirements import Requirement
from packaging.specifiers import SpecifierSet
from packaging.utils import canonicalize_name
from packaging.version import Version

project = tomllib.loads((REPO_ROOT / "pyproject.toml").read_text())
locked = tomllib.loads((REPO_ROOT / "uv.lock").read_text())
if Version(sys.version.split()[0]) not in SpecifierSet(project["project"]["requires-python"]):
    raise RuntimeError("현재 Python이 프로젝트 요구 조건과 다릅니다. 환경 A의 Python 기준을 확인하세요.")
if any(hashlib.sha256((REPO_ROOT / name).read_bytes()).hexdigest() != digest
       for name, digest in ENV_REFERENCE_SHA256.items()):
    raise RuntimeError("환경 기준 파일이 바뀌었습니다. 커널을 재시작하고 환경 A부터 실행하세요.")

# OS별 잠금 항목을 선택한다. +cu130 접미사도 비교하여 CPU 빌드를 걸러낸다.
required_names = {canonicalize_name(Requirement(spec).name)
                  for spec in project["project"]["dependencies"]} | {"pandas"}
version_rows = []
for name in sorted(required_names):
    expected = {p["version"] for p in locked["package"]
                if canonicalize_name(p["name"]) == name and "version" in p
                and (not p.get("resolution-markers")
                     or any(Marker(marker).evaluate() for marker in p["resolution-markers"]))}
    if len(expected) != 1:
        raise RuntimeError(f"현재 OS의 {name} 잠금 버전을 하나로 결정할 수 없습니다. uv.lock과 실행 환경을 확인하세요.")
    expected_version = next(iter(expected))
    try:
        actual_version = metadata.version(name)
    except metadata.PackageNotFoundError:
        actual_version = "미설치"
    version_rows.append({"package": name, "locked": expected_version, "installed": actual_version,
                         "match": actual_version != "미설치" and Version(actual_version) == Version(expected_version)})
mismatches = [row for row in version_rows if not row["match"]]
if mismatches:
    raise RuntimeError(f"잠금 버전과 다른 패키지가 있습니다: {mismatches}. "
                       "Colab은 환경 B, 로컬은 uv sync --locked 실행 후 커널을 재시작하세요.")

# 설치 메타데이터가 맞아도 메모리에는 이전 패키지가 남아 있을 수 있다.
module_names = {"numpy": "numpy", "pandas": "pandas", "pillow": "PIL", "matplotlib": "matplotlib",
                "torch": "torch", "torchvision": "torchvision", "torchmetrics": "torchmetrics", "wandb": "wandb"}
for package, module_name in module_names.items():
    loaded_version = getattr(sys.modules.get(module_name), "__version__", None)
    if loaded_version is not None and Version(loaded_version) != Version(metadata.version(package)):
        raise RuntimeError(f"{package}의 설치 버전과 메모리 버전이 다릅니다. 커널 재시작 후 환경 A부터 실행하세요.")

import torch
import torchvision
from torchvision.ops import nms

if IS_COLAB:
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU를 사용할 수 없습니다. Colab의 하드웨어 가속기를 GPU로 선택하고 다시 연결하세요.")
    try:
        boxes = torch.tensor([[0., 0., 10., 10.], [1., 1., 9., 9.]], device="cuda")
        scores = torch.tensor([0.9, 0.8], device="cuda")
        keep = nms(boxes, scores, 0.5)
        torch.cuda.synchronize()
        if keep.cpu().tolist() != [0]:
            raise RuntimeError("NMS 결과가 예상과 다릅니다.")
        del boxes, scores, keep
    except Exception as exc:
        raise RuntimeError("CUDA 또는 torchvision GPU 연산 확인에 실패했습니다. GPU/드라이버가 잠금 CUDA 빌드를 지원하는지 확인하세요. "
                           "환경 B 적용 후에도 실패하면 다른 Colab GPU 런타임이 필요합니다.") from exc
    print("CUDA·torchvision NMS GPU 연산: 통과")
else:
    print("로컬 환경: CUDA 검사는 Colab에서 수행합니다. 실제 학습 전 셀 7의 GPU 스모크를 통과해야 합니다.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ENVIRONMENT = {"python": sys.version.split()[0], "python_executable": sys.executable,
               "packages": {row["package"]: row["installed"] for row in version_rows},
               "cuda": torch.version.cuda,
               "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
               "uv_lock_sha256": ENV_REFERENCE_SHA256["uv.lock"],
               "environment_repository_commit": ENV_REPOSITORY_COMMIT}
print("Python:", ENVIRONMENT["python"], "| torch CUDA:", ENVIRONMENT["cuda"], "| GPU:", ENVIRONMENT["device"])
display(pd.DataFrame(version_rows))


### 내장 함수 A · bbox·그리기·코드 식별자

여기부터 실제 함수 구현이다. 별도 공용 Python 파일이 필요하지 않다.

In [ ]:
# 이 셀부터 함수 구현을 노트북 안에 직접 정의한다. 별도 프로젝트 모듈을 읽지 않는다.
import ast
import hashlib
import inspect
import json
import linecache
import textwrap
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

NOTEBOOK_CODE_BLOCKS = {}


def _register_notebook_block(name):
    """실행한 셀의 소스를 기록하여 재개 검증과 Cascade 작업 전달에 사용한다.

    Args:
        name (str): 이 노트북 안의 코드 묶음 이름.
    """
    frame = inspect.currentframe().f_back
    try:
        source = "".join(linecache.getlines(frame.f_code.co_filename))
        if not source:
            shell = get_ipython()
            source = shell.history_manager.input_hist_raw[-1]
        if not source.strip():
            raise RuntimeError("실행 셀의 소스를 찾을 수 없습니다. Jupyter/Colab에서 내장 함수 셀부터 다시 실행하세요.")
        NOTEBOOK_CODE_BLOCKS[name] = source
    finally:
        del frame


def _notebook_source_hash(names):
    """셀 번호·주석을 제외한 실제 내장 코드의 식별자를 계산한다.

    Args:
        names (list): 해시할 코드 묶음 이름.

    Returns:
        str: Python 3.11과 3.12에서 공통으로 비교할 수 있는 소스 해시.
    """
    entries = {}
    for name in names:
        if name not in NOTEBOOK_CODE_BLOCKS:
            raise RuntimeError(f"내장 함수 셀이 빠졌습니다: {name}. 위에서부터 함수 셀을 실행하세요.")
        tree = ast.parse(NOTEBOOK_CODE_BLOCKS[name])
        for node in ast.walk(tree):
            # 이 코드에서는 제네릭 타입 문법을 쓰지 않으므로 3.12의 빈 필드만 제외한다.
            if hasattr(node, "type_params") and not node.type_params:
                del node.type_params
        entries[name] = ast.dump(tree, include_attributes=False)
    return hashlib.sha256(json.dumps(entries, sort_keys=True, ensure_ascii=False).encode()).hexdigest()


def common_code_identity():
    """세 노트북에 같은 내용으로 들어간 데이터·평가 함수의 해시를 반환한다.

    Returns:
        str: 비교에 사용할 공통 소스 식별자.
    """
    return _notebook_source_hash(["helpers", "data", "analysis"])


def xyxy_to_xywh(bbox):
    """Ultralytics 추론 결과의 xyxy bbox를 xywh로 바꾼다.

    Args:
        bbox (list[float]): [좌상단 x, 좌상단 y, 우하단 x, 우하단 y]. 원본 픽셀.

    Returns:
        list[float]: [좌상단 x, 좌상단 y, 너비, 높이]. 원본 픽셀.
    """
    x1, y1, x2, y2 = bbox
    return [x1, y1, x2 - x1, y2 - y1]

def draw_boxes(image, boxes, labels=None, color="red", width=4):
    """이미지에 xywh bbox를 그려 새 PIL 이미지로 돌려준다.

    기본 글꼴은 한글을 그리지 못하므로 labels에는 category_id 같은 숫자·영문을 넣는다.

    Args:
        image (str | Path | PIL.Image.Image): 이미지 경로나 PIL 이미지.
        boxes (list[list[float]]): 그릴 bbox 목록. 각각 [좌상단 x, 좌상단 y, 너비, 높이], 원본 픽셀.
        labels (list | None): bbox마다 위에 적을 글자. boxes와 순서가 같다. None이면 적지 않는다.
        color (str): 선과 글자의 색.
        width (int): 선 두께(픽셀).

    Returns:
        PIL.Image.Image: bbox를 그린 RGB 이미지.
    """
    image = Image.open(image) if not isinstance(image, Image.Image) else image
    image = image.convert("RGB")
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default(size=28)
    for i, (x, y, w, h) in enumerate(boxes):
        draw.rectangle([x, y, x + w, y + h], outline=color, width=width)
        if labels:
            draw.text((x, max(0, y - 32)), str(labels[i]), fill=color, font=font)
    return image

_register_notebook_block("helpers")

### 내장 함수 B · 데이터·고정 분할·수정 라벨

Faster·Cascade·비교 노트북에 같은 코드를 넣는다.

In [ ]:
"""알약 검출의 고정 분할·수정 라벨·원본 좌표를 관리한다."""

import ast
import hashlib
import json
import os
import shutil
import tempfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision.ops import box_convert
from torchvision.transforms.functional import to_tensor


def file_sha256(path):
    """파일 내용을 해시한다.

    Args:
        path (str | Path): 검사할 파일.

    Returns:
        str: SHA-256.
    """
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def stable_hash(value):
    """JSON으로 표현 가능한 설정의 식별자를 만든다.

    Args:
        value (object): 해시할 값.

    Returns:
        str: SHA-256.
    """
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                     allow_nan=False).encode()).hexdigest()


def write_json(path, value):
    """임시 파일을 완성한 뒤 원자적으로 JSON을 저장한다.

    Args:
        path (str | Path): 저장 위치.
        value (object): JSON 값.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    text = json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False)
    with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=path.parent,
                                     prefix=path.name, delete=False) as stream:
        temporary = Path(stream.name)
        stream.write(text)
    os.replace(temporary, path)


def _bbox(value):
    if isinstance(value, str):
        value = ast.literal_eval(value)
    array = np.asarray(value, dtype=float)
    if array.shape != (4,) or not np.isfinite(array).all() or (array[2:] <= 0).any():
        raise ValueError("bbox는 유한한 xywh 숫자 4개와 양수 너비·높이여야 합니다. 라벨표를 수정하세요.")
    return array.tolist()


def prepare_data(image_dir, annotation_dir, split_path, fixes_path):
    """원본은 보존하고 고정 분할과 add/modify 수정표를 적용한다.

    Args:
        image_dir (str | Path): 학습 이미지 폴더.
        annotation_dir (str | Path): COCO JSON 폴더.
        split_path (str | Path): file_name, combo, split 열의 고정 분할.
        fixes_path (str | Path): file_name, category_id, action, bbox 열의 수정표.

    Returns:
        dict: 이미지 목록·클래스 매핑·객체 라벨·데이터 식별자.
    """
    image_dir, annotation_dir = Path(image_dir), Path(annotation_dir)
    split_path, fixes_path = Path(split_path), Path(fixes_path)
    for path in (image_dir, annotation_dir, split_path, fixes_path):
        if not path.exists():
            raise FileNotFoundError(f"데이터 경로가 없습니다: {path}. 노트북 1번의 경로를 지정하세요.")
    json_paths = sorted(annotation_dir.rglob("*.json"))
    if not json_paths:
        raise ValueError("annotation JSON이 없습니다. train_annotations 경로를 확인하세요.")
    images, annotations, names, source_hashes = {}, [], {}, []
    for path in json_paths:
        doc = json.loads(path.read_text(encoding="utf-8-sig"))
        local_images = {int(item["id"]): item for item in doc["images"]}
        for item in local_images.values():
            name = item["file_name"]
            if Path(name).name != name:
                raise ValueError("이미지 파일명에 경로가 포함됐습니다. 이미지 manifest를 확인하세요.")
            record = {"file_name": name, "image_id": int(item["id"]),
                      "width": int(item["width"]), "height": int(item["height"])}
            if name in images and images[name] != record:
                raise ValueError(f"같은 이미지의 ID·크기가 다릅니다: {name}. 원본 연결을 확인하세요.")
            images[name] = record
        for category in doc.get("categories", []):
            names[int(category["id"])] = str(category.get("name") or category["id"])
        for ann in doc.get("annotations", []):
            if int(ann["image_id"]) not in local_images:
                raise ValueError(f"JSON 내부 image_id가 연결되지 않습니다: {path}. 라벨을 확인하세요.")
            annotations.append({"file_name": local_images[int(ann["image_id"])]["file_name"],
                                "category_id": int(ann["category_id"]), "bbox": _bbox(ann["bbox"]),
                                "iscrowd": int(ann.get("iscrowd", 0))})
        source_hashes.append((str(path.relative_to(annotation_dir)), file_sha256(path)))
    if not annotations:
        raise ValueError("객체 라벨이 없습니다. annotation 내용을 확인하세요.")
    raw_count = len(annotations)
    fixes = pd.read_csv(fixes_path, keep_default_na=False)
    if "bbox" not in fixes.columns and "bbox_after" in fixes.columns:
        fixes = fixes.rename(columns={"bbox_after": "bbox"})
    required = {"file_name", "category_id", "action", "bbox"}
    if not required.issubset(fixes.columns):
        raise ValueError(f"수정표에 {sorted(required)} 열이 필요합니다. label_fixes.csv를 확인하세요.")
    changed = Counter()
    for row in fixes.to_dict("records"):
        name, category = str(row["file_name"]).strip(), int(row["category_id"])
        if name not in images:
            raise ValueError(f"수정할 이미지가 원본에 없습니다: {name}. 수정표를 확인하세요.")
        bbox, action = _bbox(row["bbox"]), str(row["action"]).strip().lower()
        # 팀 공유표의 bbox·area도 기존 객체 수정으로 정규화한다.
        action = {"bbox": "modify", "area": "modify"}.get(action, action)
        matches = [a for a in annotations if a["file_name"] == name and a["category_id"] == category]
        if action == "add":
            if any(np.allclose(a["bbox"], bbox, atol=1e-6, rtol=0) for a in matches):
                raise ValueError(f"추가 라벨이 중복됩니다: {name}, {category}. 수정표를 확인하세요.")
            annotations.append({"file_name": name, "category_id": category, "bbox": bbox, "iscrowd": 0})
        elif action == "modify":
            if len(matches) != 1:
                raise ValueError(f"수정 대상이 {len(matches)}개입니다: {name}, {category}. 라벨을 구분하세요.")
            matches[0]["bbox"] = bbox
        else:
            raise ValueError(f"지원하지 않는 수정 action: {action}. add·modify·bbox·area로 지정하세요.")
        changed[action] += 1
    frame = pd.DataFrame(sorted(images.values(), key=lambda r: r["file_name"]))
    if frame["image_id"].duplicated().any():
        raise ValueError("서로 다른 이미지에 같은 image_id가 있습니다. 원본 연결을 확인하세요.")
    split = pd.read_csv(split_path, dtype=str, keep_default_na=False)
    if not {"file_name", "combo", "split"}.issubset(split.columns):
        raise ValueError("분할표에 file_name·combo·split 열이 필요합니다. 고정 분할표를 확인하세요.")
    if split["file_name"].duplicated().any() or set(split["file_name"]) != set(frame["file_name"]):
        raise ValueError("분할표의 이미지 목록이 원본과 다릅니다. 누락·중복을 확인하세요.")
    if set(split["split"]) != {"train", "val"}:
        raise ValueError("분할에는 train·val이 모두 있어야 합니다. 고정 분할표를 확인하세요.")
    if not split["combo"].eq(split["file_name"].str.split("_").str[0]).all():
        raise ValueError("분할표의 combo와 파일명 조합이 다릅니다. 고정 분할표를 확인하세요.")
    if split.groupby("combo")["split"].nunique().max() > 1:
        raise ValueError("같은 조합이 train·val에 섞였습니다. 팀의 고정 분할표를 확인하세요.")
    frame = frame.merge(split, on="file_name", validate="one_to_one")
    image_hashes = []
    for row in frame.to_dict("records"):
        path = image_dir / row["file_name"]
        with Image.open(path) as image:
            if image.size != (row["width"], row["height"]):
                raise ValueError(f"이미지와 라벨의 크기가 다릅니다: {path}. 좌표 기준을 확인하세요.")
        image_hashes.append((row["file_name"], file_sha256(path)))
    hash_splits = defaultdict(set)
    for name, digest in image_hashes:
        hash_splits[digest].add(split.set_index("file_name").loc[name, "split"])
    if any(len(parts) > 1 for parts in hash_splits.values()):
        raise ValueError("동일한 이미지 내용이 train·val에 중복됩니다. 고정 분할을 검토하세요.")
    categories = sorted({a["category_id"] for a in annotations})
    mapping = {category: index + 1 for index, category in enumerate(categories)}
    objects = defaultdict(list)
    for ann in annotations:
        info = images[ann["file_name"]]
        x, y, w, h = ann["bbox"]
        if x < 0 or y < 0 or x + w > info["width"] or y + h > info["height"]:
            raise ValueError(f"경계 밖 bbox: {ann['file_name']}, {ann['category_id']}. 수정표를 확인하세요.")
        if ann["iscrowd"] != 0:
            raise ValueError("이 실험은 개별 알약 iscrowd=0을 전제로 합니다. 평가 정책을 확인하세요.")
        xyxy = box_convert(torch.tensor([ann["bbox"]], dtype=torch.float64), "xywh", "xyxy")[0].tolist()
        objects[ann["file_name"]].append({"boxes": xyxy, "category_id": ann["category_id"],
                                          "label": mapping[ann["category_id"]]})
    class_map = pd.DataFrame({"category_id": categories,
                              "model_label": [mapping[c] for c in categories],
                              "name": [names.get(c, str(c)) for c in categories]})
    identity = stable_hash({"images": image_hashes, "annotations": source_hashes,
                            "split": file_sha256(split_path), "fixes": file_sha256(fixes_path),
                            "mapping": mapping})
    return {"frame": frame, "objects": dict(objects), "class_map": class_map,
            "category_to_label": mapping, "label_to_category": {v: k for k, v in mapping.items()},
            "image_dir": image_dir, "identity": identity, "image_hashes": dict(image_hashes),
            "raw_boxes": raw_count, "fixed_boxes": len(annotations), "fixes": dict(changed)}


def data_summary(bundle):
    """분할별 이미지 수와 클래스별 정답 객체 수를 집계한다.

    Args:
        bundle (dict): prepare_data 결과.

    Returns:
        tuple: 분할 요약표, 클래스별 객체 수.
    """
    rows = []
    for row in bundle["frame"].to_dict("records"):
        for obj in bundle["objects"].get(row["file_name"], []):
            rows.append({"split": row["split"], "category_id": obj["category_id"]})
    counts = pd.crosstab(pd.Series([r["category_id"] for r in rows], name="category_id"),
                         pd.Series([r["split"] for r in rows], name="split"))
    counts = counts.reindex(index=bundle["class_map"]["category_id"], columns=["train", "val"], fill_value=0)
    summary = pd.DataFrame([{"split": part, "images": int(bundle["frame"]["split"].eq(part).sum()),
                             "boxes": int(counts[part].sum()), "classes": int((counts[part] > 0).sum())}
                            for part in ("train", "val")])
    return summary, counts.reset_index()


def cache_images(bundle, cache_root):
    """내용 해시가 일치하는 로컬 이미지 캐시를 준비한다.

    Args:
        bundle (dict): 데이터 묶음.
        cache_root (str | Path): 복사본 저장 루트.

    Returns:
        dict: image_dir만 캐시 경로로 바꾼 묶음.
    """
    destination = Path(cache_root) / bundle["identity"][:12]
    destination.mkdir(parents=True, exist_ok=True)
    for name, digest in bundle["image_hashes"].items():
        target = destination / name
        if not target.exists() or file_sha256(target) != digest:
            with tempfile.NamedTemporaryFile(dir=destination, delete=False) as stream:
                temporary = Path(stream.name)
            shutil.copyfile(bundle["image_dir"] / name, temporary)
            if file_sha256(temporary) != digest:
                raise IOError("이미지 복사본이 원본과 다릅니다. 캐시를 다시 준비하세요.")
            os.replace(temporary, target)
    return {**bundle, "image_dir": destination}


class PillDataset(Dataset):
    """원본 픽셀 이미지와 torchvision용 정답을 반환한다."""

    def __init__(self, frame, image_dir, objects=None):
        self.frame = frame.reset_index(drop=True).copy()
        self.image_dir = Path(image_dir)
        self.objects = objects

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(self.image_dir / row["file_name"]) as image:
            tensor = to_tensor(image.convert("RGB"))
        objects = [] if self.objects is None else self.objects.get(row["file_name"], [])
        boxes = torch.tensor([o["boxes"] for o in objects], dtype=torch.float32).reshape(-1, 4)
        return tensor, {"boxes": boxes, "labels": torch.tensor([o["label"] for o in objects], dtype=torch.int64),
                        "image_id": torch.tensor(int(row["image_id"])),
                        "area": (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]),
                        "iscrowd": torch.zeros(len(objects), dtype=torch.int64)}


def collate_detection(batch):
    """객체 수가 다른 이미지와 정답을 목록으로 묶는다.

    Args:
        batch (list): 이미지·정답 쌍.

    Returns:
        tuple: 이미지 목록, 정답 목록.
    """
    return tuple(zip(*batch))


def make_dataset(bundle, split):
    """지정된 고정 분할의 Dataset을 만든다.

    Args:
        bundle (dict): 데이터 묶음.
        split (str): train 또는 val.

    Returns:
        PillDataset: 선택 분할.
    """
    return PillDataset(bundle["frame"].loc[bundle["frame"]["split"].eq(split)],
                       bundle["image_dir"], bundle["objects"])


def cache_dataset(dataset, cache_root):
    """Test를 포함한 Dataset의 이미지를 검증된 로컬 캐시로 연결한다.

    Args:
        dataset (PillDataset): 복사할 이미지 목록.
        cache_root (str | Path): 캐시 루트.

    Returns:
        PillDataset: 같은 순서·정답을 유지하는 로컬 Dataset.
    """
    hashes = {name: file_sha256(dataset.image_dir / name) for name in dataset.frame["file_name"]}
    source = {"image_dir": dataset.image_dir, "image_hashes": hashes,
              "identity": stable_hash({"files": hashes, "frame": dataset.frame.to_dict("records")})}
    cached = cache_images(source, cache_root)
    return PillDataset(dataset.frame, cached["image_dir"], dataset.objects)


def test_dataset(image_dir):
    """숫자 파일명을 대회 image_id로 보존한 Test Dataset을 만든다.

    Args:
        image_dir (str | Path): Test 이미지 폴더.

    Returns:
        PillDataset: 정답이 없는 이미지 목록.
    """
    image_dir = Path(image_dir)
    rows = []
    for path in sorted(image_dir.iterdir()):
        if path.suffix.lower() not in {".png", ".jpg", ".jpeg"}:
            continue
        if not path.stem.isdigit():
            raise ValueError(f"Test 파일명이 숫자가 아닙니다: {path.name}. 대회 파일을 확인하세요.")
        with Image.open(path) as image:
            width, height = image.size
        rows.append({"file_name": path.name, "image_id": int(path.stem), "width": width, "height": height})
    frame = pd.DataFrame(rows)
    if frame.empty or frame["image_id"].duplicated().any():
        raise ValueError("Test 이미지가 없거나 image_id가 중복됩니다. 경로·파일명을 확인하세요.")
    return PillDataset(frame.sort_values("image_id"), image_dir)


_register_notebook_block("data")

### 내장 함수 C · mAP·오류 분석·시각화·CSV

IoU 0.75~0.95를 0.05 간격으로 평가한다. 세 노트북이 동일한 평가 함수를 사용한다.

In [ ]:
"""모델에 독립적인 원본 좌표 평가·오류 분석·제출 변환."""

from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torchvision.ops import box_iou, box_convert




IOU_THRESHOLDS = [0.75, 0.80, 0.85, 0.90, 0.95]
SUBMISSION_COLUMNS = ["annotation_id", "image_id", "category_id", "bbox_x", "bbox_y",
                      "bbox_w", "bbox_h", "score"]


def evaluate_records(records, class_map=None, detailed=True):
    """공통 예측의 bbox mAP와 선택적 클래스·IoU별 AP를 계산한다.

    Args:
        records (list): prediction과 target을 포함한 이미지별 결과. labels는 원래 category ID.
        class_map (DataFrame | None): 전체 클래스 목록.
        detailed (bool): 상세 결과 계산 여부.

    Returns:
        dict: 요약, 클래스별 AP, IoU별 AP, PR 곡선 원자료.
    """
    if not records or any("target" not in row for row in records):
        raise ValueError("정답이 있는 validation 결과가 필요합니다. 4번 검증 추론을 실행하세요.")
    metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox", iou_thresholds=IOU_THRESHOLDS,
                                  max_detection_thresholds=[1, 10, 100], class_metrics=detailed,
                                  extended_summary=detailed)
    metric.update([row["prediction"] for row in records], [row["target"] for row in records])
    result = metric.compute()
    summary = {"map_75_95": float(result["map"]), "map_75": float(result["map_75"]),
               "mar_100": float(result["mar_100"]), "images": len(records),
               "ground_truths": sum(len(r["target"]["boxes"]) for r in records)}
    if not detailed:
        return {"summary": summary}
    counts = Counter(int(c) for row in records for c in row["target"]["labels"])
    precision = result["precision"][:, :, :, 0, -1].numpy()
    classes = result["classes"].reshape(-1).tolist()

    def valid_mean(values):
        valid = values[values >= 0]
        return float(valid.mean()) if valid.size else None

    ap = pd.DataFrame([{"category_id": int(category), "ap_75_95": valid_mean(precision[:, :, index]),
                        "val_objects": counts[int(category)]} for index, category in enumerate(classes)])
    if class_map is not None:
        ap = class_map[["category_id", "name"]].merge(ap, how="left", on="category_id")
        ap["val_objects"] = ap["val_objects"].fillna(0).astype(int)
    ap.loc[ap["val_objects"].eq(0), "ap_75_95"] = np.nan
    iou_ap = pd.DataFrame({"iou": IOU_THRESHOLDS,
                           "ap": [valid_mean(precision[index]) for index in range(len(IOU_THRESHOLDS))]})
    return {"summary": summary, "classes": ap, "ious": iou_ap,
            "precision": precision, "category_ids": classes}


def detection_errors(records, categories, iou_threshold=0.75, score_threshold=0.30):
    """클래스와 독립적인 IoU 일대일 대응으로 혼동과 미검출을 집계한다.

    Args:
        records (list): validation 공통 결과.
        categories (list): 원래 category ID 목록.
        iou_threshold (float): 대응 최소 IoU.
        score_threshold (float): 진단용 최소 score.

    Returns:
        tuple: 행=정답·열=예측 혼동행렬, 오류 상세 표.
    """
    if not 0 < iou_threshold <= 1 or not 0 <= score_threshold <= 1:
        raise ValueError("진단 IoU·score 범위가 잘못됐습니다. 4번 설정을 확인하세요.")
    categories = [int(c) for c in categories]
    positions = {category: index for index, category in enumerate(categories)}
    background = len(categories)
    matrix = np.zeros((background + 1, background + 1), dtype=int)
    events = []
    for record in records:
        target, pred = record["target"], record["prediction"]
        kept = torch.where(pred["scores"] >= score_threshold)[0]
        boxes, labels, scores = pred["boxes"][kept], pred["labels"][kept], pred["scores"][kept]
        overlaps = box_iou(target["boxes"], boxes).numpy()
        candidates = np.argwhere(overlaps >= iou_threshold)
        candidates = sorted(candidates.tolist(), key=lambda pair: (-overlaps[tuple(pair)], pair))
        used_gt, used_pred = set(), set()

        def append(kind, gt=None, prediction=None, overlap=None):
            events.append({"image_id": int(record["image_id"]), "file_name": record["file_name"],
                           "kind": kind, "gt_index": gt,
                           "pred_index": None if prediction is None else int(kept[prediction]),
                           "true_category": None if gt is None else int(target["labels"][gt]),
                           "pred_category": None if prediction is None else int(labels[prediction]),
                           "score": None if prediction is None else float(scores[prediction]),
                           "iou": overlap})

        for gt, prediction in candidates:
            if gt in used_gt or prediction in used_pred:
                continue
            used_gt.add(gt)
            used_pred.add(prediction)
            truth, estimate = int(target["labels"][gt]), int(labels[prediction])
            matrix[positions[truth], positions[estimate]] += 1
            append("correct" if truth == estimate else "class_confusion", gt, prediction,
                   float(overlaps[gt, prediction]))
        for gt in range(len(target["boxes"])):
            if gt not in used_gt:
                matrix[positions[int(target["labels"][gt])], background] += 1
                possible = [p for p in range(len(boxes)) if int(labels[p]) == int(target["labels"][gt])
                            and 0 < overlaps[gt, p] < iou_threshold]
                best = max(possible, key=lambda p: overlaps[gt, p]) if possible else None
                append("localization_or_missed" if best is not None else "missed", gt, best,
                       None if best is None else float(overlaps[gt, best]))
        for prediction in range(len(boxes)):
            if prediction not in used_pred:
                matrix[background, positions[int(labels[prediction])]] += 1
                duplicate = any(int(target["labels"][gt]) == int(labels[prediction])
                                and overlaps[gt, prediction] >= iou_threshold for gt in used_gt)
                append("duplicate" if duplicate else "extra", prediction=prediction)
    labels = [str(c) for c in categories] + ["unmatched"]
    columns = ["image_id", "file_name", "kind", "gt_index", "pred_index", "true_category",
               "pred_category", "score", "iou"]
    return pd.DataFrame(matrix, index=labels, columns=labels), pd.DataFrame(events, columns=columns)


def plot_ground_truth(dataset, reverse, count=3):
    """입력 데이터의 정답 bbox를 표시한다.

    Args:
        dataset (PillDataset): 정답이 있는 이미지.
        reverse (dict): 모델 번호에서 원래 클래스 ID로 가는 매핑.
        count (int): 표시할 이미지 수.

    Returns:
        Figure: 정답 박스 이미지.
    """
    fig, axes = plt.subplots(1, min(count, len(dataset)), figsize=(6 * min(count, len(dataset)), 6), squeeze=False)
    for index, axis in enumerate(axes[0]):
        _, target = dataset[index]
        image = draw_boxes(dataset.image_dir / dataset.frame.iloc[index]["file_name"],
                           [xyxy_to_xywh(box.tolist()) for box in target["boxes"]],
                           [str(reverse[int(c)]) for c in target["labels"]], color="lime")
        axis.imshow(image)
        axis.set(title=dataset.frame.iloc[index]["file_name"][:30])
        axis.axis("off")
    fig.tight_layout()
    return fig


def plot_learning_curves(run_dirs):
    """완료 여부와 관계없이 저장된 학습 곡선을 표시한다.

    Args:
        run_dirs (list): metrics.csv를 포함한 실행 폴더.

    Returns:
        Figure: loss·mAP 곡선.
    """
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for directory in run_dirs:
        path = Path(directory) / "metrics.csv"
        if not path.exists():
            continue
        history = pd.read_csv(path)
        if history.empty:
            continue
        name = Path(directory).name
        axes[0].plot(history["epoch"], history["train_loss"], label=name)
        axes[1].plot(history["epoch"], history["val_map_75_95"], label=name)
    for axis, title in zip(axes, ["Train loss", "Validation mAP@[.75:.95]"]):
        axis.set(title=title, xlabel="Epoch")
        axis.grid(alpha=0.25)
        if axis.lines:
            axis.legend(fontsize=7)
    fig.tight_layout()
    return fig


def plot_candidate_scores(results):
    """설정별 seed 점수와 평균·표본 표준편차를 함께 표시한다.

    Args:
        results (DataFrame): 완료된 실행 결과.

    Returns:
        Figure: seed별 점수와 설정별 평균. 단일 seed의 표준편차는 추정하지 않음.
    """
    summary = results.groupby("candidate_id")["best_val_map_75_95"].agg(["mean", "std", "count"])
    summary = summary.sort_values("mean", ascending=False)
    positions = {key: index for index, key in enumerate(summary.index)}
    x = np.arange(len(summary))
    fig, axis = plt.subplots(figsize=(max(8, len(summary)*1.2), 5))
    axis.bar(x, summary["mean"], color="lightsteelblue", label="Mean")
    estimated = summary["count"].ge(2).to_numpy()
    axis.errorbar(x[estimated], summary.loc[estimated, "mean"], yerr=summary.loc[estimated, "std"],
                  fmt="none", ecolor="black", capsize=4, label="Sample SD (n>=2)")
    for seed, group in results.groupby("seed"):
        axis.scatter([positions[key] for key in group["candidate_id"]], group["best_val_map_75_95"],
                     label=f"seed {int(seed)}", zorder=3)
    axis.set(xticks=x, xticklabels=[f"{key}\nn={int(summary.loc[key, 'count'])}" for key in summary.index],
             ylabel="Validation mAP@[.75:.95]", ylim=(0, 1.05),
             title="Candidate and seed comparison (n=1: SD unavailable)")
    axis.tick_params(axis="x", rotation=30, labelsize=8)
    axis.legend(fontsize=8)
    axis.grid(axis="y", alpha=0.2)
    fig.tight_layout()
    return fig


def plot_ap(report):
    """낮은 클래스 AP·정답 수와 IoU별 AP를 함께 표시한다.

    Args:
        report (dict): 상세 evaluate_records 결과.

    Returns:
        Figure: 클래스·IoU별 그래프.
    """
    low = report["classes"].dropna(subset=["ap_75_95"]).nsmallest(15, "ap_75_95")
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].barh([f"{int(row.category_id)} (n={int(row.val_objects)})" for row in low.itertuples()], low["ap_75_95"])
    axes[0].invert_yaxis()
    axes[0].set(title="Lowest class AP (validation support)", xlabel="AP@[.75:.95]", xlim=(0, 1))
    axes[1].plot(report["ious"]["iou"], report["ious"]["ap"], marker="o")
    axes[1].set(title="AP by IoU", xlabel="IoU", ylabel="AP", ylim=(0, 1))
    axes[1].grid(alpha=0.25)
    fig.tight_layout()
    return fig


def plot_confusion(matrix, normalize=True):
    """정답 행 기준 정규화 또는 원시 건수 혼동행렬을 표시한다.

    Args:
        matrix (DataFrame): 행=정답·열=예측. unmatched 행은 추가 예측, 열은 미검출.
        normalize (bool): 정답 행별 비율 표시 여부.

    Returns:
        Figure: 진단 혼동행렬.
    """
    values = matrix.to_numpy(dtype=float)
    if normalize:
        values = np.divide(values, values.sum(axis=1, keepdims=True),
                           out=np.zeros_like(values), where=values.sum(axis=1, keepdims=True) > 0)
    size = max(9, min(20, len(matrix) * 0.28))
    fig, axis = plt.subplots(figsize=(size, size))
    heatmap = axis.imshow(values, cmap="Blues", vmin=0, vmax=1 if normalize else None)
    axis.set(xticks=range(len(matrix)), yticks=range(len(matrix)),
             xticklabels=matrix.columns, yticklabels=matrix.index,
             xlabel="Predicted (unmatched = missed)", ylabel="True (unmatched = extra)",
             title="Detection confusion: row-normalized" if normalize else "Detection confusion: counts")
    axis.tick_params(axis="x", rotation=90, labelsize=7)
    axis.tick_params(axis="y", labelsize=7)
    fig.colorbar(heatmap, ax=axis, fraction=0.04)
    fig.tight_layout()
    return fig


def preview_records(records, events=None, count=6, score_threshold=0.30):
    """정답·예측과 오류 종류를 표시한다. Test에는 정답을 그리지 않는다.

    Args:
        records (list): 공통 예측 결과.
        events (DataFrame | None): validation 오류 표.
        count (int): 최대 이미지 수.
        score_threshold (float): 표시용 점수 하한.

    Returns:
        Figure: 초록 정답·빨강 예측 이미지.
    """
    if events is not None:
        errors = events.loc[events["kind"].ne("correct")].copy()
        errors = errors.sort_values("score", ascending=False, na_position="last")
        ids = list(dict.fromkeys(errors["image_id"].tolist()))[:count]
        chosen = [r for identifier in ids for r in records if r["image_id"] == identifier]
    else:
        chosen = records[:count]
    if not chosen:
        fig, axis = plt.subplots(figsize=(8, 2))
        axis.text(0.5, 0.5, "No error images at the chosen diagnostic thresholds", ha="center")
        axis.axis("off")
        return fig
    fig, axes = plt.subplots(len(chosen), 1, figsize=(9, 6 * len(chosen)), squeeze=False)
    for record, axis in zip(chosen, axes[:, 0]):
        with Image.open(record["image_path"]) as source:
            image = source.convert("RGB")
        if "target" in record:
            target = record["target"]
            image = draw_boxes(image, [xyxy_to_xywh(b.tolist()) for b in target["boxes"]],
                               [f"GT:{int(c)}" for c in target["labels"]], color="lime")
        pred = record["prediction"]
        indices = torch.where(pred["scores"] >= score_threshold)[0]
        indices = indices[torch.argsort(pred["scores"][indices], descending=True)][:20]
        captions = []
        for index in indices:
            caption = f"P:{int(pred['labels'][index])} {float(pred['scores'][index]):.2f}"
            if "target" in record and len(record["target"]["boxes"]):
                overlap = float(box_iou(pred["boxes"][index].reshape(1, 4), record["target"]["boxes"]).max())
                caption += f" I:{overlap:.2f}"
            captions.append(caption)
        image = draw_boxes(image, [xyxy_to_xywh(pred["boxes"][i].tolist()) for i in indices],
                           captions, color="red")
        image.thumbnail((1280, 1280))
        kinds = "prediction only; no ground truth"
        if events is not None:
            kinds = ", ".join(sorted(set(events.loc[events["image_id"].eq(record["image_id"]), "kind"])))
        axis.imshow(image)
        axis.set_title(f"ID {record['image_id']} | {kinds}\ngreen=GT, red=prediction; score>={score_threshold}", fontsize=9)
        axis.axis("off")
    fig.tight_layout()
    return fig


def submission_frame(records, allowed_categories):
    """공통 원본 좌표 예측을 대회 CSV 표로 변환한다.

    Args:
        records (list): labels가 원래 category ID인 공통 예측.
        allowed_categories (list): 허용 클래스 목록.

    Returns:
        DataFrame: 대회 컬럼 순서의 표. 빈 예측도 빈 표로 표현한다.
    """
    allowed_categories = set(int(c) for c in allowed_categories)
    if len({int(r["image_id"]) for r in records}) != len(records):
        raise ValueError("예측 이미지 ID가 중복됩니다. 전체 추론 목록을 확인하세요.")
    rows = []
    for record in records:
        pred = record["prediction"]
        if not (len(pred["boxes"]) == len(pred["labels"]) == len(pred["scores"])):
            raise ValueError("예측 boxes·labels·scores 수가 다릅니다. 공통 변환을 확인하세요.")
        for box, category, score in zip(pred["boxes"], pred["labels"], pred["scores"]):
            x, y, w, h = xyxy_to_xywh(box.tolist())
            if int(category) not in allowed_categories:
                raise ValueError("알 수 없는 category ID입니다. 모델의 클래스 매핑을 확인하세요.")
            if not np.isfinite([x, y, w, h, float(score)]).all() or not 0 <= float(score) <= 1:
                raise ValueError("유한하지 않은 좌표·score가 있습니다. 모델 출력을 확인하세요.")
            if x < -1e-4 or y < -1e-4 or w <= 0 or h <= 0 or x+w > record["width"]+1e-4 or y+h > record["height"]+1e-4:
                raise ValueError("원본 경계 밖 또는 면적 0 예측입니다. 좌표 복원을 확인하세요.")
            rows.append({"image_id": int(record["image_id"]), "category_id": int(category),
                         "bbox_x": x, "bbox_y": y, "bbox_w": w, "bbox_h": h, "score": float(score)})
    frame = pd.DataFrame(rows, columns=SUBMISSION_COLUMNS[1:])
    frame = frame.sort_values(["image_id", "score"], ascending=[True, False], kind="stable").reset_index(drop=True)
    frame.insert(0, "annotation_id", np.arange(1, len(frame)+1, dtype=np.int64))
    for column in ("annotation_id", "image_id", "category_id"):
        frame[column] = frame[column].astype("int64")
    return frame[SUBMISSION_COLUMNS]


def restore_submission(frame, records):
    """CSV를 다시 읽은 뒤 이미지별 공통 예측으로 복원한다.

    Args:
        frame (DataFrame): 제출 컬럼의 표.
        records (list): 전체 이미지·정답 목록.

    Returns:
        list: 빈 이미지도 보존한 복원 결과.
    """
    if list(frame.columns) != SUBMISSION_COLUMNS or frame.isna().any().any():
        raise ValueError("제출 컬럼·결측값이 잘못됐습니다. CSV 생성 코드를 확인하세요.")
    try:
        frame = frame.apply(pd.to_numeric, errors="raise")
    except (TypeError, ValueError) as error:
        raise ValueError("CSV에 숫자가 아닌 값이 있습니다. 제출 파일을 다시 생성하세요.") from error
    if not np.isfinite(frame.to_numpy(dtype=float)).all():
        raise ValueError("CSV에 유한하지 않은 값이 있습니다. 제출 파일을 다시 생성하세요.")
    for column in ("annotation_id", "image_id", "category_id"):
        if not np.equal(frame[column], np.floor(frame[column])).all():
            raise ValueError("CSV ID는 정수여야 합니다. 클래스·이미지 매핑을 확인하세요.")
    if frame["annotation_id"].duplicated().any() or not set(frame["image_id"]).issubset({r["image_id"] for r in records}):
        raise ValueError("제출 ID가 중복되거나 이미지 목록과 다릅니다. CSV를 확인하세요.")
    restored = []
    for record in records:
        rows = frame.loc[frame["image_id"].eq(record["image_id"])]
        boxes = torch.tensor(rows[["bbox_x", "bbox_y", "bbox_w", "bbox_h"]].to_numpy(dtype=np.float32), dtype=torch.float32).reshape(-1, 4)
        restored.append({**record, "prediction": {
            "boxes": box_convert(boxes, "xywh", "xyxy"),
            "labels": torch.tensor(rows["category_id"].to_numpy(dtype=np.int64), dtype=torch.int64),
            "scores": torch.tensor(rows["score"].to_numpy(dtype=np.float32), dtype=torch.float32)}})
    return restored


def verify_roundtrip(records, categories, csv_path):
    """실제 CSV 저장·읽기와 전후 mAP·좌표·ID 일치를 검사한다.

    Args:
        records (list): 정답을 포함한 validation 예측.
        categories (list): 원래 클래스 ID 목록.
        csv_path (str | Path): 검증용 CSV 저장 위치.

    Returns:
        dict: 변환 전후 mAP와 통과 결과.
    """
    path = Path(csv_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frame = submission_frame(records, categories)
    frame.to_csv(path, index=False)
    loaded = pd.read_csv(path)
    restored = restore_submission(loaded, records)
    for before, after in zip(records, restored):
        left, right = before["prediction"], after["prediction"]
        order = torch.argsort(left["scores"], descending=True, stable=True)
        if not torch.equal(left["labels"][order], right["labels"]) or not torch.allclose(left["boxes"][order], right["boxes"], atol=1e-3, rtol=0) or not torch.allclose(left["scores"][order], right["scores"], atol=1e-7, rtol=0):
            raise ValueError("CSV 복원 전후 예측이 바뀌었습니다. ID·좌표 변환을 확인하세요.")
    before = evaluate_records(records, detailed=False)["summary"]["map_75_95"]
    after = evaluate_records(restored, detailed=False)["summary"]["map_75_95"]
    if abs(before-after) > 1e-6:
        raise ValueError("CSV 변환 전후 mAP가 다릅니다. 평가 입력과 좌표 변환을 확인하세요.")
    report = {"before": before, "after": after, "passed": True, "images": len(records), "rows": len(frame)}
    write_json(path.with_suffix(".json"), report)
    return report


_register_notebook_block("analysis")

### 내장 함수 D · Faster R-CNN 학습·저장·재개

COCO 사전학습·FE/PT/FT·AdamW·W&B·예측과 비교 자료 저장을 구현한다.

In [ ]:
"""COCO Faster R-CNN의 세 학습 범위·9개 후보·반복 실험을 실행한다."""

import gc
import importlib.metadata
import json
import math
import os
import random
import sys
import tempfile
import time
import uuid
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights, fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import FrozenBatchNorm2d




DEFAULT_CONFIG = {
    "max_epochs": 100, "patience": 30, "image_size": 640, "batch_size": 16,
    "weight_decay": 1e-4, "betas": [0.9, 0.999], "warmup_epochs": 3,
    "final_lr_ratio": 0.01, "num_workers": 0, "score_threshold": 0.001,
    "nms_iou": 0.5, "max_detections": 100, "preview_score": 0.30,
    "preview_count": 3, "experiment_tag": "fix", "wandb_project": "pill-object-detection",
    "wandb_entity": None, "wandb_mode": "online", "use_wandb": True,
    "require_cuda": True, "deterministic": True, "precision": "fp32",
    "normalization": "FrozenBatchNorm2d_eps0_fixed_statistics_and_affine",
    "augmentation": "none", "weights": "COCO_V1", "optimizer": "AdamW",
    "learning_rates": [1e-4, 3e-4, 1e-3],
}
MODES = ("ft", "pt", "fe")
LEARNING_RATES = (1e-4, 3e-4, 1e-3)


def runtime_info():
    """실제로 import한 학습 환경을 기록한다.

    Returns:
        dict: 패키지·CUDA·GPU 정보.
    """
    versions = {name: importlib.metadata.version(name) for name in
                ("torch", "torchvision", "torchmetrics", "numpy", "pandas", "wandb", "pycocotools", "pillow", "matplotlib")}
    return {"packages": versions, "python": sys.version.split()[0], "cuda": torch.version.cuda,
            "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
            "framework": "torchvision", "model": "faster_rcnn_r50_fpn",
            "uv_lock_sha256": globals().get("ENV_REFERENCE_SHA256", {}).get("uv.lock")}


def code_identity():
    """별도 Python 파일 없이 실제 실행한 내장 함수의 해시를 반환한다.

    Returns:
        dict: 공통 코드와 해당 모델 학습 코드의 소스 식별자.
    """
    names = ["helpers", "data", "analysis", "training"]
    if "cascade_model" in NOTEBOOK_CODE_BLOCKS:
        names.append("cascade_model")
    return {name: _notebook_source_hash([name]) for name in names}


def validate_config(config):
    """지원하는 설정과 수치 범위를 검사한다.

    Args:
        config (dict): DEFAULT_CONFIG에 기반한 실험 설정.
    """
    if set(config) != set(DEFAULT_CONFIG):
        raise ValueError("설정 키가 기본 규격과 다릅니다. DEFAULT_CONFIG를 복사해 수정하세요.")
    for key in ("max_epochs", "patience", "image_size", "batch_size", "preview_count"):
        if not isinstance(config[key], int) or config[key] <= 0:
            raise ValueError(f"{key}는 양의 정수여야 합니다. 1번 설정을 확인하세요.")
    if config["weights"] != "COCO_V1" or config["optimizer"] != "AdamW" or config["precision"] != "fp32":
        raise ValueError("이번 비교는 COCO_V1·AdamW·FP32만 지원합니다. 1번 설정을 확인하세요.")
    if config["augmentation"] != "none" or config["normalization"] != DEFAULT_CONFIG["normalization"]:
        raise ValueError("증강·정규화 변경은 별도 구현이 필요합니다. 지원하는 공통 정책을 사용하세요.")
    if not 0 <= config["warmup_epochs"] < config["max_epochs"] or not 0 < config["final_lr_ratio"] <= 1:
        raise ValueError("warmup·최종 학습률 비율이 잘못됐습니다. 1번 설정을 확인하세요.")
    if not 0 <= config["score_threshold"] <= 1 or not 0 < config["nms_iou"] <= 1 or config["max_detections"] != 100:
        raise ValueError("score·NMS 또는 평가 max detections가 잘못됐습니다. 최대 100개 정책을 확인하세요.")
    if config["weight_decay"] < 0 or len(config["betas"]) != 2 or any(not 0 <= v < 1 for v in config["betas"]):
        raise ValueError("AdamW 설정이 잘못됐습니다. decay·betas를 확인하세요.")
    if config["num_workers"] < 0 or config["wandb_mode"] not in {"online", "offline", "disabled"}:
        raise ValueError("workers 또는 W&B mode가 잘못됐습니다. 1번 설정을 확인하세요.")
    if not config["experiment_tag"] or any(c not in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-" for c in config["experiment_tag"]):
        raise ValueError("실험 태그는 영문·숫자·밑줄·하이픈으로 지정하세요.")
    if len(config["learning_rates"]) != 3 or len(set(config["learning_rates"])) != 3 or any(not math.isfinite(v) or v <= 0 for v in config["learning_rates"]):
        raise ValueError("서로 다른 양수 학습률 3개를 지정하세요. 1번 설정의 learning_rates를 확인하세요.")


def seed_everything(seed, deterministic=True):
    """분할을 변경하지 않고 학습 난수만 설정한다.

    Args:
        seed (int): 학습 seed.
        deterministic (bool): 결정적 연산 사용 여부.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = deterministic
    torch.use_deterministic_algorithms(deterministic)


def lr_text(lr):
    """학습률을 읽기 쉬운 이름으로 표현한다.

    Args:
        lr (float): 학습률.

    Returns:
        str: 예를 들어 3e-4.
    """
    mantissa, exponent = f"{lr:.8e}".split("e")
    return f"{mantissa.rstrip('0').rstrip('.')}e{int(exponent)}"


def experiment_spec(mode, lr, seed):
    """학습 범위·학습률·seed로 실험을 식별한다.

    Args:
        mode (str): ft, pt, fe.
        lr (float): 학습률.
        seed (int): 0, 1, 2.

    Returns:
        dict: 후보 ID와 실행 조건.
    """
    if mode not in MODES or not math.isfinite(lr) or lr <= 0 or seed not in (0, 1, 2):
        raise ValueError("실험은 ft·pt·fe, 양수 학습률, seed 0·1·2로 지정하세요.")
    candidate = f"{mode}_lr{lr_text(lr)}"
    return {"candidate_id": candidate, "spec_id": f"{candidate}_s{seed}",
            "mode": mode, "lr": float(lr), "seed": int(seed)}


def candidate_table(learning_rates=LEARNING_RATES):
    """사전학습 기반 3개 방식과 학습률 3개를 조합한다.

    Args:
        learning_rates (list | tuple): 설정 셀에서 지정한 학습률 후보 3개.

    Returns:
        DataFrame: seed 0 후보 9개.
    """
    return pd.DataFrame([experiment_spec(mode, lr, 0) for mode in MODES for lr in learning_rates])


def run_name(config, spec):
    """사용자 지정 접두부와 조건을 포함한 W&B 이름을 만든다.

    Args:
        config (dict): 공통 설정.
        spec (dict): 실험 조건.

    Returns:
        str: cj_faster_e100_s1_fix_ft_lr3e-4 형식.
    """
    return (f"cj_faster_e{config['max_epochs']}_s{spec['seed']}_{config['experiment_tag']}_"
            f"{spec['mode']}_lr{lr_text(spec['lr'])}")


def _recipe(config, bundle):
    validate_config(config)
    return {"config": config, "data_identity": bundle["identity"], "code": code_identity(),
            "runtime": runtime_info(), "common_code": common_code_identity(), "class_mapping": {str(k): v for k, v in bundle["category_to_label"].items()}}


def experiment_root(root, config, bundle):
    """이번 설계의 코드·환경·데이터·설정과 일치하는 결과만 모은다.

    Args:
        root (str | Path): 결과 루트.
        config (dict): 공통 설정.
        bundle (dict): 데이터 묶음.

    Returns:
        Path: 실험 묶음 경로.
    """
    return Path(root) / stable_hash(_recipe(config, bundle))[:12]


def _frozen_normalization(module):
    for name, child in list(module.named_children()):
        if isinstance(child, torch.nn.BatchNorm2d):
            frozen = FrozenBatchNorm2d(child.num_features, eps=0.0)
            for attribute in ("weight", "bias", "running_mean", "running_var"):
                getattr(frozen, attribute).copy_(getattr(child, attribute).detach())
            setattr(module, name, frozen)
        elif isinstance(child, FrozenBatchNorm2d):
            child.eps = 0.0
        else:
            _frozen_normalization(child)


def build_model(num_classes, spec, config, load_coco=True):
    """같은 COCO 검출 모델에서 출력층과 학습 범위를 설정한다.

    Args:
        num_classes (int): 배경을 제외한 클래스 수.
        spec (dict): 실험 조건.
        config (dict): 공통 설정.
        load_coco (bool): 신규 학습은 True. 저장된 전체 체크포인트 복원 시에만 False.

    Returns:
        Module: CPU의 Faster R-CNN. 모든 정규화 통계·계수는 고정.
    """
    validate_config(config)
    experiment_spec(spec["mode"], spec["lr"], spec["seed"])
    weights = FasterRCNN_ResNet50_FPN_Weights.COCO_V1 if load_coco else None
    model = fasterrcnn_resnet50_fpn(weights=weights, weights_backbone=None,
                                    trainable_backbone_layers=5 if load_coco else None,
                                    min_size=config["image_size"], max_size=config["image_size"])
    _frozen_normalization(model)
    features = model.roi_heads.box_predictor.cls_score.in_features
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(spec["seed"])
        model.roi_heads.box_predictor = FastRCNNPredictor(features, num_classes + 1)
    for parameter in model.parameters():
        parameter.requires_grad_(True)
    if spec["mode"] != "ft":
        model.backbone.requires_grad_(False)
        if spec["mode"] == "pt":
            model.backbone.body.layer4.requires_grad_(True)
            model.backbone.fpn.requires_grad_(True)
    model.roi_heads.score_thresh = config["score_threshold"]
    model.roi_heads.nms_thresh = config["nms_iou"]
    model.roi_heads.detections_per_img = config["max_detections"]
    return model


def parameter_summary(model):
    """실제 학습 모듈과 파라미터 수를 확인한다.

    Args:
        model (Module): 검사 모델.

    Returns:
        DataFrame: 모듈별 전체·학습 가능한 파라미터 수.
    """
    groups = {}
    for name, parameter in model.named_parameters():
        group = ".".join(name.split(".")[:3]) if name.startswith("backbone.body") else name.split(".")[0]
        row = groups.setdefault(group, {"module": group, "total": 0, "trainable": 0})
        row["total"] += parameter.numel()
        row["trainable"] += parameter.numel() if parameter.requires_grad else 0
    return pd.DataFrame(groups.values())


def make_optimizer(model, spec, config):
    """고정 파라미터를 제외하고 AdamW decay 그룹을 만든다.

    Args:
        model (Module): 학습 모델.
        spec (dict): 학습률 조건.
        config (dict): AdamW 설정.

    Returns:
        AdamW: bias·1차원 파라미터에는 decay 미적용.
    """
    groups = [[], []]
    for name, parameter in model.named_parameters():
        if parameter.requires_grad:
            groups[int(name.endswith("bias") or parameter.ndim <= 1)].append(parameter)
    return torch.optim.AdamW([{"params": groups[0], "weight_decay": config["weight_decay"]},
                               {"params": groups[1], "weight_decay": 0.0}],
                              lr=spec["lr"], betas=tuple(config["betas"]))


def make_scheduler(optimizer, steps_per_epoch, config):
    """optimizer step 단위의 선형 warmup·cosine 스케줄러를 만든다.

    Args:
        optimizer (Optimizer): AdamW.
        steps_per_epoch (int): 실제 epoch당 batch 수.
        config (dict): 학습 예산·warmup 설정.

    Returns:
        LambdaLR: 첫 step부터 warmup을 적용하는 scheduler.
    """
    total = config["max_epochs"] * steps_per_epoch
    warmup = int(config["warmup_epochs"] * steps_per_epoch)

    def factor(step):
        if warmup and step < warmup:
            return (step + 1) / warmup
        progress = min(1.0, max(0.0, (step-warmup) / max(1, total-warmup-1)))
        return config["final_lr_ratio"] + (1-config["final_lr_ratio"]) * (1+math.cos(math.pi*progress))/2

    return torch.optim.lr_scheduler.LambdaLR(optimizer, factor)


def _device(config):
    if config["require_cuda"] and not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU가 없습니다. Colab CUDA 커널을 선택하고 1번부터 실행하세요.")
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _loader(dataset, config, training=False, generator=None):
    return DataLoader(dataset, batch_size=config["batch_size"] if training else 1,
                      shuffle=training, num_workers=config["num_workers"],
                      collate_fn=collate_detection, generator=generator,
                      pin_memory=torch.cuda.is_available())


def _synchronize(device):
    if device.type == "cuda":
        torch.cuda.synchronize()


def collect_predictions(model, dataset, config, reverse, with_targets=True):
    """추론을 한 번 수행해 원본 category ID의 공통 결과를 만든다.

    Args:
        model (Module): 복원 또는 학습 모델.
        dataset (PillDataset): 순서가 고정된 이미지 목록.
        config (dict): 추론 설정.
        reverse (dict): 모델 번호→원래 category ID.
        with_targets (bool): validation 정답 포함 여부.

    Returns:
        tuple: 공통 결과, 순수 추론 시간·전체 시간.
    """
    device = next(model.parameters()).device
    records, model_seconds = [], 0.0
    start_all = time.perf_counter()
    model.eval()
    with torch.inference_mode():
        for images, targets in _loader(dataset, config):
            inputs = [image.to(device) for image in images]
            _synchronize(device)
            start = time.perf_counter()
            outputs = model(inputs)
            _synchronize(device)
            model_seconds += time.perf_counter()-start
            for output, target in zip(outputs, targets):
                info = dataset.frame.iloc[len(records)].to_dict()
                prediction = {key: output[key].detach().cpu() for key in ("boxes", "labels", "scores")}
                prediction["labels"] = torch.tensor([reverse[int(c)] for c in prediction["labels"]], dtype=torch.int64)
                record = {key: int(info[key]) for key in ("image_id", "width", "height")}
                record.update(file_name=info["file_name"], image_path=str(dataset.image_dir / info["file_name"]),
                              prediction=prediction)
                if with_targets:
                    record["target"] = {"boxes": target["boxes"].cpu(),
                                        "labels": torch.tensor([reverse[int(c)] for c in target["labels"]], dtype=torch.int64)}
                records.append(record)
    return records, {"model_ms_per_image": 1000*model_seconds/max(1, len(records)),
                     "wall_seconds": time.perf_counter()-start_all, "images": len(records)}


def smoke_test(bundle, config, spec):
    """실제 batch·입력·정밀도·학습 범위로 첫 AdamW step을 검사한다.

    Args:
        bundle (dict): 학습 데이터 묶음.
        config (dict): 본 학습과 동일한 설정.
        spec (dict): 검사할 실험 조건.

    Returns:
        dict: 설정·데이터·GPU에 연결된 통과 증명과 파라미터 요약.
    """

    import matplotlib.pyplot as plt
    device = _device(config)
    seed_everything(spec["seed"], config["deterministic"])
    dataset = make_dataset(bundle, "train")
    if len(dataset) < config["batch_size"]:
        raise ValueError("실제 batch보다 학습 이미지가 적습니다. 데이터·공통 batch를 확인하세요.")
    model = build_model(len(bundle["class_map"]), spec, config).to(device)
    table = parameter_summary(model)
    optimizer = make_optimizer(model, spec, config)
    before = {name: p.detach().cpu().clone() for name, p in model.named_parameters()}
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    try:
        model.train()
        images, targets = next(iter(_loader(dataset, config, True)))
        losses = model([image.to(device) for image in images],
                       [{k: v.to(device) for k, v in target.items()} for target in targets])
        loss = sum(losses.values())
        if not torch.isfinite(loss):
            raise FloatingPointError("스모크 loss가 유한하지 않습니다. 이미지·라벨·설정을 확인하세요.")
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        if any(p.grad is not None and not torch.isfinite(p.grad).all() for p in model.parameters()):
            raise FloatingPointError("스모크 gradient가 유한하지 않습니다. 입력·학습률을 확인하세요.")
        optimizer.step()
        changed, frozen_ok = False, True
        for name, parameter in model.named_parameters():
            same = torch.equal(before[name], parameter.detach().cpu())
            if parameter.requires_grad:
                changed |= not same
            else:
                frozen_ok &= same
        if not changed or not frozen_ok or not optimizer.state:
            raise RuntimeError("학습·고정 범위 또는 optimizer 갱신에 문제가 있습니다. 파라미터 설정을 확인하세요.")
        validation = make_dataset(bundle, "val")
        validation = PillDataset(validation.frame.head(1), validation.image_dir, validation.objects)
        records, _ = collect_predictions(model, validation, config, bundle["label_to_category"], True)
        converted = submission_frame(records[:1], bundle["category_to_label"])
        figure = preview_records(records[:1], count=1, score_threshold=config["preview_score"])
        try:
            figure.canvas.draw()
        finally:
            plt.close(figure)
        report = {"passed": True, "signature": stable_hash({"recipe": _recipe(config, bundle), "spec": spec}),
                  "loss": float(loss.detach()), "batch_size": len(images),
                  "peak_gpu_gb": torch.cuda.max_memory_allocated()/1024**3 if device.type == "cuda" else None,
                  "trainable_changed": changed, "frozen_unchanged": frozen_ok,
                  "visualization_checked": True, "submission_rows": len(converted),
                  "parameters": table.to_dict("records")}
        return report
    finally:
        del model, optimizer, before
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()


def _atomic_save(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=path.parent, delete=False) as stream:
        temporary = Path(stream.name)
    torch.save(value, temporary)
    os.replace(temporary, path)


def _rng_state(generator):
    return {"python": random.getstate(), "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(), "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
            "loader": generator.get_state()}


def _restore_rng(state, generator):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if state["cuda"]:
        if len(state["cuda"]) != torch.cuda.device_count():
            raise RuntimeError("재개 시 GPU 구성이 다릅니다. 같은 구성에서 재개하세요.")
        torch.cuda.set_rng_state_all(state["cuda"])
    generator.set_state(state["loader"])


def completed_results(root, config, bundle):
    """이번 설계에서 완료된 실행만 읽는다. 과거 베이스라인은 읽지 않는다.

    Args:
        root (str | Path): 새 실험 전용 결과 루트.
        config (dict): 공통 설정.
        bundle (dict): 데이터 묶음.

    Returns:
        DataFrame: 완료 실행 표.
    """
    folder = experiment_root(root, config, bundle)
    rows = [json.loads(path.read_text()) for path in sorted(folder.glob("*/result.json"))]
    columns = ["candidate_id", "spec_id", "mode", "lr", "seed", "best_val_map_75_95", "best_epoch",
               "last_epoch", "training_seconds", "inference_ms", "peak_gpu_gb", "run_name", "run_dir"]
    return pd.DataFrame(rows).reindex(columns=columns)


def select_candidates(results, candidates, destination, top_k=2, selected_ids=None):
    """후보 탐색 완료 후 validation 기준 상위 설정을 저장한다.

    Args:
        results (DataFrame): 완료 결과.
        candidates (DataFrame): seed 0 후보 표.
        destination (str | Path): 선택 결과 JSON.
        top_k (int): 선택 개수.
        selected_ids (list | None): 직접 지정할 후보. None이면 mAP 순.

    Returns:
        list: seed 1·2에 사용할 실험 조건.
    """
    explored = results.loc[results["seed"].eq(0)].copy()
    if set(explored["candidate_id"]) != set(candidates["candidate_id"]):
        raise ValueError("seed 0의 후보 9개가 미완료입니다. 나머지 후보를 학습한 뒤 선택하세요.")
    if explored["candidate_id"].duplicated().any():
        raise ValueError("같은 후보 결과가 여러 개입니다. 사용할 run을 명시한 뒤 선택하세요.")
    ranked = explored.sort_values(["best_val_map_75_95", "training_seconds", "candidate_id"], ascending=[False, True, True])
    identifiers = list(ranked["candidate_id"].head(top_k)) if selected_ids is None else list(selected_ids)
    if len(identifiers) != top_k or len(set(identifiers)) != top_k or not set(identifiers).issubset(set(candidates["candidate_id"])):
        raise ValueError("상위 후보 ID가 잘못됐습니다. 서로 다른 후보 2개를 지정하세요.")
    selected = [candidates.set_index("candidate_id").loc[key].to_dict() for key in identifiers]
    repeat = [experiment_spec(row["mode"], row["lr"], seed) for row in selected for seed in (1, 2)]
    write_json(destination, {"basis": "validation_only", "candidate_ids": identifiers, "repeat_specs": repeat})
    return repeat


def run_experiment(bundle, config, spec, root, smoke_report=None, resume_dir=None):
    """필수 스모크 후 독립 학습 또는 명시적 체크포인트 재개를 수행한다.

    Args:
        bundle (dict): 고정 데이터.
        config (dict): 공통 설정.
        spec (dict): 학습 방식·학습률·seed.
        root (str | Path): 새 실험 결과 루트.
        smoke_report (dict | None): 일치하는 설정의 스모크 증명. 없거나 다르면 다시 검사.
        resume_dir (str | Path | None): 재개할 실행 폴더. 자동 재개하지 않음.

    Returns:
        dict: 완료 실행 결과.
    """
    recipe = _recipe(config, bundle)
    signature = stable_hash({"recipe": recipe, "spec": spec})
    folder = experiment_root(root, config, bundle)
    existing = completed_results(root, config, bundle)
    match = existing.loc[existing["spec_id"].eq(spec["spec_id"])]
    if resume_dir is None and not match.empty:
        if len(match) != 1:
            raise ValueError("같은 조건 결과가 여러 개입니다. 결과 목록에서 사용할 run을 명시하세요.")
        print("완료된 이번 실험 결과 재사용:", match.iloc[0]["run_name"])
        return match.iloc[0].to_dict()
    if not smoke_report or not smoke_report.get("passed") or smoke_report.get("signature") != signature:
        print("본 학습 설정으로 스모크 테스트:", spec["spec_id"], flush=True)
        smoke_report = smoke_test(bundle, config, spec)
    device = _device(config)
    seed_everything(spec["seed"], config["deterministic"])
    identifier = uuid.uuid4().hex[:12]
    name = run_name(config, spec)
    run_dir = Path(resume_dir) if resume_dir is not None else folder / f"{name}__{identifier}"
    if resume_dir is None:
        run_dir.mkdir(parents=True, exist_ok=False)
        manifest = {"signature": signature, "recipe": recipe, "spec": spec, "run_id": identifier,
                    "run_name": name, "run_dir": str(run_dir.resolve())}
        write_json(run_dir / "config.json", manifest)
        bundle["class_map"].to_csv(run_dir / "class_map.csv", index=False)
    else:
        manifest = json.loads((run_dir / "config.json").read_text())
        if manifest["signature"] != signature or (run_dir / "result.json").exists():
            raise ValueError("재개 설정이 다르거나 이미 완료된 실행입니다. 재개할 폴더를 확인하세요.")
    write_json(run_dir / "smoke.json", smoke_report)
    generator = torch.Generator().manual_seed(spec["seed"])
    train_loader = _loader(make_dataset(bundle, "train"), config, True, generator)
    model = build_model(len(bundle["class_map"]), spec, config, load_coco=resume_dir is None).to(device)
    optimizer = make_optimizer(model, spec, config)
    scheduler = make_scheduler(optimizer, len(train_loader), config)
    best, best_epoch, stale, start_epoch, history, previous_seconds = -1.0, 0, 0, 1, [], 0.0
    resume_state = None
    if resume_dir is not None:
        resume_state = torch.load(run_dir / "last.pt", map_location="cpu", weights_only=False)
        if resume_state["signature"] != signature:
            raise ValueError("체크포인트 설정이 다릅니다. 같은 실험 폴더를 지정하세요.")
        model.load_state_dict(resume_state["model"], strict=True)
        optimizer.load_state_dict(resume_state["optimizer"])
        scheduler.load_state_dict(resume_state["scheduler"])
        best, best_epoch, stale = resume_state["best"], resume_state["best_epoch"], resume_state["stale"]
        start_epoch, history = resume_state["epoch"]+1, resume_state["history"]
        previous_seconds = resume_state["training_seconds"]
        pd.DataFrame(history).to_csv(run_dir / "metrics.csv", index=False)
    wandb_run = None
    try:
        if config["use_wandb"]:
            import wandb
            wandb_run = wandb.init(project=config["wandb_project"], entity=config["wandb_entity"],
                                   name=name, id=manifest["run_id"], resume="must" if resume_dir else "never",
                                   mode=config["wandb_mode"], dir=str(run_dir), config=manifest,
                                   group=folder.name, tags=["cj", "faster", spec["mode"], config["experiment_tag"]])
        if resume_state is not None:
            _restore_rng(resume_state["rng"], generator)
        else:
            # 로깅 초기화가 학습 난수에 영향을 주지 않게 다시 설정한다.
            seed_everything(spec["seed"], config["deterministic"])
        if device.type == "cuda":
            torch.cuda.reset_peak_memory_stats()
        training_start = time.perf_counter()
        peak = float(resume_state.get("peak_gpu_gb", 0)) if resume_state else 0.0
        # 종료 직후 result.json 저장 전에 중단된 실행은 반복 학습하지 않는다.
        epochs = range(start_epoch, config["max_epochs"]+1) if stale < config["patience"] else []
        for epoch in epochs:
            epoch_start = time.perf_counter()
            model.train()
            losses_total, seen = {}, 0
            first_lr = optimizer.param_groups[0]["lr"]
            for images, targets in train_loader:
                loss_dict = model([image.to(device) for image in images],
                                  [{key: value.to(device) for key, value in target.items()} for target in targets])
                loss = sum(loss_dict.values())
                if not torch.isfinite(loss):
                    raise FloatingPointError("loss가 유한하지 않습니다. 입력·라벨·학습률을 확인하세요.")
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
                scheduler.step()
                for key, value in loss_dict.items():
                    losses_total[key] = losses_total.get(key, 0.0) + float(value.detach())*len(images)
                seen += len(images)
            records, speed = collect_predictions(model, make_dataset(bundle, "val"), config,
                                                  bundle["label_to_category"], True)
            score = evaluate_records(records, detailed=False)["summary"]["map_75_95"]
            if not math.isfinite(score) or score < 0:
                raise RuntimeError("validation mAP를 계산할 수 없습니다. 정답·클래스 연결을 확인하세요.")
            improved = score > best
            if improved:
                best, best_epoch, stale = score, epoch, 0
            else:
                stale += 1
            peak = max(peak, torch.cuda.max_memory_allocated()/1024**3 if device.type == "cuda" else 0.0)
            row = {"epoch": epoch, "train_loss": sum(losses_total.values())/seen,
                   "val_map_75_95": score, "lr": first_lr, "last_lr": optimizer.param_groups[0]["lr"],
                   "epoch_seconds": time.perf_counter()-epoch_start, "val_inference_ms": speed["model_ms_per_image"],
                   "peak_gpu_gb": peak, **{key: value/seen for key, value in losses_total.items()}}
            history.append(row)
            if wandb_run is not None:
                try:

                    import wandb
                    previews = []
                    for record in records[:config["preview_count"]]:
                        truth, pred = record["target"], record["prediction"]
                        image = draw_boxes(record["image_path"], [xyxy_to_xywh(b.tolist()) for b in truth["boxes"]],
                                           [f"GT:{int(c)}" for c in truth["labels"]], color="lime")
                        kept = torch.where(pred["scores"] >= config["preview_score"])[0][:20]
                        image = draw_boxes(image, [xyxy_to_xywh(pred["boxes"][i].tolist()) for i in kept],
                                           [f"P:{int(pred['labels'][i])} {float(pred['scores'][i]):.2f}" for i in kept], color="red")
                        image.thumbnail((1000, 1000))
                        previews.append(wandb.Image(image, caption=f"epoch {epoch} | ID {record['image_id']} | green=GT, red=prediction"))
                    wandb_run.log({**row, "Media/validation": previews}, step=epoch)
                except Exception as error:
                    print(f"W&B 이미지·기록 실패: {type(error).__name__}: {error}. 체크포인트와 로컬 지표는 계속 저장합니다.", flush=True)
            elapsed = previous_seconds+time.perf_counter()-training_start
            state = {"signature": signature, "epoch": epoch, "model": model.state_dict(),
                     "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                     "best": best, "best_epoch": best_epoch, "stale": stale, "history": history,
                     "training_seconds": elapsed, "peak_gpu_gb": peak, "rng": _rng_state(generator)}
            if improved:
                _atomic_save(run_dir / "best.pt", {"signature": signature, "epoch": epoch,
                                                    "model": model.state_dict(), "map_75_95": score})
            _atomic_save(run_dir / "last.pt", state)
            pd.DataFrame(history).to_csv(run_dir / "metrics.csv", index=False)
            print(f"{spec['spec_id']} | {epoch:03d}/{config['max_epochs']} | loss={row['train_loss']:.4f} "
                  f"| mAP={score:.4f} | patience={stale}/{config['patience']}", flush=True)
            if stale >= config["patience"]:
                break
        if not history:
            raise RuntimeError("학습 이력이 없습니다. 체크포인트와 epoch 설정을 확인하세요.")
        best_row = next(row for row in history if row["epoch"] == best_epoch)
        result = {**spec, "best_val_map_75_95": best, "best_epoch": best_epoch,
                  "last_epoch": history[-1]["epoch"], "training_seconds": previous_seconds+time.perf_counter()-training_start,
                  "inference_ms": best_row["val_inference_ms"], "peak_gpu_gb": peak,
                  "run_name": name, "run_dir": str(run_dir.resolve())}
        write_json(run_dir / "result.json", result)
        return result
    finally:
        if wandb_run is not None:
            wandb_run.finish()
        del model, optimizer, scheduler
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()


def predict_run(run_dir, dataset, bundle, config, with_targets=True, force=False):
    """저장된 best 모델의 추론을 실행하거나 동일 조건 캐시를 재사용한다.

    Args:
        run_dir (str | Path): 이번 설계의 실행 폴더.
        dataset (PillDataset): validation 또는 Test Dataset.
        bundle (dict): 학습 데이터·클래스 매핑.
        config (dict): 공통 설정.
        with_targets (bool): validation 정답 포함 여부.
        force (bool): 캐시가 일치해도 추론 재실행 여부.

    Returns:
        dict: records·추론 속도·캐시 식별자.
    """
    run_dir = Path(run_dir)
    manifest = json.loads((run_dir / "config.json").read_text())
    spec = manifest["spec"]
    if manifest["signature"] != stable_hash({"recipe": _recipe(config, bundle), "spec": spec}):
        raise ValueError("분석 대상의 코드·설정·데이터가 다릅니다. 해당 실행과 같은 환경을 준비하세요.")
    checkpoint = run_dir / "best.pt"
    image_identity = [(r.file_name, file_sha256(dataset.image_dir / r.file_name)) for r in dataset.frame.itertuples()]
    identity = stable_hash({"checkpoint": file_sha256(checkpoint), "images": image_identity,
                            "frame": dataset.frame[["file_name", "image_id", "width", "height"]].to_dict("records"),
                            "with_targets": with_targets, "signature": manifest["signature"]})
    path = run_dir / ("validation_predictions.pt" if with_targets else "test_predictions.pt")
    if path.exists() and not force:
        payload = torch.load(path, map_location="cpu", weights_only=False)
        if payload["identity"] == identity:
            # 캐시 생성 후 저장 위치가 바뀌어도 현재 이미지 경로를 사용한다.
            for record in payload["records"]:
                record["image_path"] = str(dataset.image_dir / record["file_name"])
            return payload
    model = build_model(len(bundle["class_map"]), spec, config, load_coco=False)
    state = torch.load(checkpoint, map_location="cpu", weights_only=False)
    if state["signature"] != manifest["signature"]:
        raise ValueError("best.pt와 실험 설정이 다릅니다. 같은 run의 파일을 지정하세요.")
    model.load_state_dict(state["model"], strict=True)
    model.to(_device(config))
    try:
        records, speed = collect_predictions(model, dataset, config, bundle["label_to_category"], with_targets)
        payload = {"identity": identity, "records": records, "speed": speed}
        _atomic_save(path, payload)
        return payload
    finally:
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def export_comparison_run(run_dir, dataset, bundle, config, predictor=None):
    """비교 노트북이 재학습 없이 읽을 validation 예측과 검사 정보를 저장한다.

    Args:
        run_dir (str | Path): 완료된 학습 폴더.
        dataset (PillDataset): 전체 validation Dataset.
        bundle (dict): 고정 데이터와 클래스 매핑.
        config (dict): 해당 학습 설정.
        predictor (callable | None): Cascade 전용 환경의 추론 함수. None이면 현재 모델 추론.

    Returns:
        dict: 비교 입력 파일과 데이터·환경·평가 정책.
    """
    directory = Path(run_dir)
    inference = predict_run if predictor is None else predictor
    payload = inference(directory, dataset, bundle, config, with_targets=True)
    config_doc = json.loads((directory / "config.json").read_text())
    result = json.loads((directory / "result.json").read_text())
    if len(payload["records"]) != len(dataset):
        raise ValueError("전체 validation 예측이 아닙니다. 셀 13의 추론을 다시 실행하세요.")
    # CSV 저장·복원 후에도 같은 mAP인지 확인한다.
    checked = verify_roundtrip(payload["records"], bundle["class_map"]["category_id"],
                               directory / "analysis/validation_roundtrip.csv")
    prediction_path = directory / "validation_predictions.pt"
    document = {
        "format": "pill_detection_notebook_comparison_v1",
        "data_identity": bundle["identity"], "common_code": common_code_identity(),
        "class_map": bundle["class_map"].to_dict("records"),
        "iou_thresholds": IOU_THRESHOLDS, "max_detections": config["max_detections"],
        "score_threshold": config["score_threshold"],
        "prediction_file": prediction_path.name, "prediction_sha256": file_sha256(prediction_path),
        "run_signature": config_doc["signature"], "spec": config_doc["spec"],
        "runtime": config_doc["recipe"]["runtime"], "config": config_doc["recipe"]["config"],
        "speed": payload["speed"], "roundtrip": checked,
        "model": config_doc["recipe"]["runtime"]["model"],
        "result": {key: value for key, value in result.items() if key != "run_dir"},
    }
    write_json(directory / "comparison_manifest.json", document)
    return {"run_dir": str(directory), "seed": document["spec"]["seed"],
            "manifest": str(directory / "comparison_manifest.json"), "map_75_95": checked["after"]}


_register_notebook_block("training")

### 셀 2 · 원본·수정 라벨·고정 분할·결과 경로


In [ ]:
# Colab이면 Drive를 연결한다. 로컬 실행은 저장소의 data를 사용한다.
IS_COLAB = Path("/content").is_dir() and "google.colab" in sys.modules
if Path("/content").is_dir():
    try:
        from google.colab import drive
        IS_COLAB = True
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
    except ImportError:
        pass

DATA_ROOT = (Path("/content/drive/MyDrive/코드잇 ;)/Project1_pill-object-detection/data")
             if IS_COLAB else REPO_ROOT / "data")
IMAGE_DIR = DATA_ROOT / "raw/train_images"
ANNOTATION_DIR = DATA_ROOT / "raw/train_annotations"
SPLIT_PATH = DATA_ROOT / "processed/split.csv"
FIXES_PATH = DATA_ROOT / "processed/label_fixes.csv"
# 팀 공유 수정표는 bbox_after 열도 지원한다. 선택된 파일 경로를 확인한다.
if not FIXES_PATH.is_file():
    FIXES_PATH = REPO_ROOT / "notebooks/eda/label_fix/jaeone_label_fixes.csv"
OUTPUT_ROOT = ((DATA_ROOT.parent if IS_COLAB else REPO_ROOT) / "runs/cj_faster_experiments")
SUBMISSION_ROOT = (DATA_ROOT.parent if IS_COLAB else REPO_ROOT) / "submissions/cj_faster_experiments"
CACHE_ROOT = Path("/content/cj_faster_image_cache") if IS_COLAB else None

paths = {"images": IMAGE_DIR, "annotations": ANNOTATION_DIR, "split": SPLIT_PATH, "fixes": FIXES_PATH}
display(pd.DataFrame([{"item": k, "path": str(v), "exists": v.exists()} for k, v in paths.items()]))
if not all(p.exists() for p in paths.values()):
    raise FileNotFoundError("데이터 파일이 없습니다. 위 경로를 수정한 뒤 이 셀부터 다시 실행하세요.")
print("새 실험 결과:", OUTPUT_ROOT)
print("제출 파일:", SUBMISSION_ROOT)

### 셀 3 · 공통 설정 한 곳에서 지정


In [ ]:
CONFIG = {
    **DEFAULT_CONFIG,
    "max_epochs": 100,
    "patience": 30,
    "image_size": 640,
    "batch_size": 16,
    "weight_decay": 1e-4,
    "learning_rates": [1e-4, 3e-4, 1e-3],
    "warmup_epochs": 3,
    "final_lr_ratio": 0.01,
    "num_workers": 0,
    "experiment_tag": "fix",
    "wandb_project": "pill-object-detection",
    "wandb_entity": "codeit-prj1-team2",
    "use_wandb": True,
    "wandb_mode": "online",
    "require_cuda": True,
}
validate_config(CONFIG)
print("사전학습: COCO_V1 | AdamW | FP32 | AMP 미사용 | 추가 랜덤 증강 없음")
print("정규화: FrozenBatchNorm의 통계·계수 모두 고정; 학습 범위와 별도로 관리")
print("평가: bbox mAP@[0.75, 0.80, 0.85, 0.90, 0.95]")
print("이름 예시:", run_name(CONFIG, experiment_spec("ft", 3e-4, 1)))
display(pd.DataFrame([{"setting": k, "value": str(v)} for k, v in CONFIG.items()]))

## 2. 데이터 준비·스모크 테스트

### 셀 4 · 고정 분할과 수정표 적용


In [ ]:
bundle = prepare_data(IMAGE_DIR, ANNOTATION_DIR, SPLIT_PATH, FIXES_PATH)
if CACHE_ROOT is not None:
    bundle = cache_images(bundle, CACHE_ROOT)
summary, support = data_summary(bundle)
print("원본 bbox:", bundle["raw_boxes"], "| 수정 후:", bundle["fixed_boxes"], "| 수정:", bundle["fixes"])
print("데이터 식별자:", bundle["identity"][:12])
display(summary)
RUN_GROUP_DIR = experiment_root(OUTPUT_ROOT, CONFIG, bundle)
RUN_GROUP_DIR.mkdir(parents=True, exist_ok=True)
write_json(RUN_GROUP_DIR / "settings.json", {"config": CONFIG, "environment": ENVIRONMENT,
                                          "data_identity": bundle["identity"]})

### 셀 5 · 클래스 번호표·Dataset 확인


In [ ]:
train_dataset = make_dataset(bundle, "train")
val_dataset = make_dataset(bundle, "val")
display(bundle["class_map"].merge(support, on="category_id", how="left"))
sample_image, sample_target = train_dataset[0]
print("이미지:", tuple(sample_image.shape), sample_image.dtype)
print("정답:", {k: tuple(v.shape) for k, v in sample_target.items()})
print("모델 번호 0은 배경, 약은 1..N. 공통 평가·제출은 원래 category ID 사용")
del sample_image, sample_target

### 셀 6 · 정답 bbox 시각화


In [ ]:
figure = plot_ground_truth(train_dataset, bundle["label_to_category"], count=3)
plt.show()
plt.close(figure)

스모크는 본 실험과 같은 batch 16·640·FP32·AdamW로 한 step을 검사한다.
본 학습에는 스모크에서 바뀐 가중치를 사용하지 않는다. 조건이 바뀌면 학습 함수가 다시 검사한다.

### 셀 7 · 본 학습 전 필수 스모크


In [ ]:
SMOKE_SPEC = experiment_spec("ft", 3e-4, 0)
SMOKE_REPORT = smoke_test(bundle, CONFIG, SMOKE_SPEC)
display(pd.DataFrame(SMOKE_REPORT["parameters"]))
print({k: v for k, v in SMOKE_REPORT.items() if k != "parameters"})

## 3. 학습·실험 관리

### 셀 8 · 후보 9개 생성


In [ ]:
CANDIDATES = candidate_table(CONFIG["learning_rates"])
CANDIDATES["run_name"] = [run_name(CONFIG, row) for row in CANDIDATES.to_dict("records")]
display(CANDIDATES)
print("seed 0 탐색 9회 + 상위 설정 2개의 seed 1·2 추가 4회 = 총 13회")

### 셀 9 · 실행할 후보 선택·W&B 인증


In [ ]:
# 먼저 후보 하나로 학습→분석→CSV 연결을 확인한 뒤 목록을 확장한다.
SELECTED_CANDIDATE_IDS = ["ft_lr3e-4"]
RUN_CANDIDATES = True
# 재개하려면 실행할 후보는 하나만 선택하고 해당 run 폴더의 절대 경로를 입력한다.
RESUME_DIR = None

selected = CANDIDATES.loc[CANDIDATES["candidate_id"].isin(SELECTED_CANDIDATE_IDS)].copy()
if len(selected) != len(set(SELECTED_CANDIDATE_IDS)) or selected.empty:
    raise ValueError("후보 ID가 잘못됐습니다. 셀 8의 candidate_id에서 선택하세요.")
if RESUME_DIR is not None and len(selected) != 1:
    raise ValueError("재개는 후보 하나씩 진행하세요. SELECTED_CANDIDATE_IDS를 수정하세요.")
display(selected)
if RUN_CANDIDATES and CONFIG["use_wandb"] and CONFIG["wandb_mode"] == "online":
    import wandb
    wandb.login()  # 키는 숨김 입력 또는 환경변수로 인증하고 출력·파일에 기록하지 않는다.
else:
    print("학습을 시작하려면 이 셀에서 RUN_CANDIDATES=True로 바꾼 뒤 셀 10을 실행하세요.")

### 셀 10 · 선택한 후보 학습·명시적 재개


In [ ]:
CONFIG["wandb_entity"] = "codeit-prj1-team2"

# 변경된 설정에 맞춰 결과 저장 위치도 갱신
RUN_GROUP_DIR = experiment_root(OUTPUT_ROOT, CONFIG, bundle)
RUN_GROUP_DIR.mkdir(parents=True, exist_ok=True)

write_json(RUN_GROUP_DIR / "settings.json", {
    "config": CONFIG,
    "environment": ENVIRONMENT,
    "data_identity": bundle["identity"],
})

print("W&B 저장 대상:", CONFIG["wandb_entity"])

In [ ]:
if RUN_CANDIDATES:
    for row in selected.to_dict("records"):
        spec = experiment_spec(row["mode"], row["lr"], row["seed"])
        result = run_experiment(bundle, CONFIG, spec, OUTPUT_ROOT,
                                smoke_report=SMOKE_REPORT, resume_dir=RESUME_DIR)
        display(pd.DataFrame([result]))
else:
    print("본 학습 미실행: 셀 9의 실행 설정을 확인하세요.")
results = completed_results(OUTPUT_ROOT, CONFIG, bundle)
display(results)

### 셀 11 · seed 0 상위 후보 2개 선택


In [ ]:
# None이면 validation mAP 순. 직접 고르면 후보 ID 두 개와 선택 근거를 기록한다.
SELECTED_TOP_IDS = None
SELECTION_REASON = "validation mAP 우선, 동일 점수일 때 학습 시간 참고"
results = completed_results(OUTPUT_ROOT, CONFIG, bundle)
display(results.loc[results["seed"].eq(0)].sort_values("best_val_map_75_95", ascending=False))
if set(results.loc[results["seed"].eq(0), "candidate_id"]) == set(CANDIDATES["candidate_id"]):
    repeat_specs = select_candidates(results, CANDIDATES, RUN_GROUP_DIR / "selection.json",
                                     selected_ids=SELECTED_TOP_IDS)
    selection = json.loads((RUN_GROUP_DIR / "selection.json").read_text())
    selection["reason"] = SELECTION_REASON
    write_json(RUN_GROUP_DIR / "selection.json", selection)
    display(pd.DataFrame(repeat_specs))
else:
    missing = sorted(set(CANDIDATES["candidate_id"]) - set(results.loc[results["seed"].eq(0), "candidate_id"]))
    print("아직 탐색하지 않은 후보:", missing)

### 셀 12 · 선택한 설정에 seed 1·2 추가


In [ ]:
# 동일 조건으로 저장된 seed 0 결과 확인
results = completed_results(OUTPUT_ROOT, CONFIG, bundle)
display(results)

seed0 = results[
    results["candidate_id"].eq("ft_lr3e-4")
    & results["seed"].eq(0)
]
if seed0.empty:
    raise RuntimeError(
        "동일 조건의 seed 0 결과를 찾지 못했습니다. "
        "저장 경로·설정·GPU가 이전과 같은지 확인하세요."
    )

import wandb
wandb.login()

# 각각 COCO 사전학습 가중치에서 독립적으로 시작
for seed in [1, 2]:
    spec = experiment_spec("ft", 3e-4, seed)
    result = run_experiment(bundle, CONFIG, spec, OUTPUT_ROOT)
    display(pd.DataFrame([result]))

results = completed_results(OUTPUT_ROOT, CONFIG, bundle)
display(results)

In [ ]:
# RUN_REPEATS = False
# if RUN_REPEATS:
#     selection_path = RUN_GROUP_DIR / "selection.json"
#     if not selection_path.is_file():
#         raise RuntimeError("상위 후보 선택이 없습니다. 셀 11을 먼저 실행하세요.")
#     selection = json.loads(selection_path.read_text())
#     for spec in selection["repeat_specs"]:
#         result = run_experiment(bundle, CONFIG, spec, OUTPUT_ROOT)
#         display(pd.DataFrame([result]))
# else:
#     print("반복 학습 미실행: 상위 후보가 정해지면 RUN_REPEATS=True로 실행하세요.")
# results = completed_results(OUTPUT_ROOT, CONFIG, bundle)
# display(results)

## 4. 검증·오류 분석

### 셀 13 · best 모델의 validation 전체 예측


In [ ]:
# None이면 완료된 이번 실험 중 최고 validation 실행을 분석한다.
ANALYSIS_RUN_DIR = None
results = completed_results(OUTPUT_ROOT, CONFIG, bundle)
validation_payload = None
if ANALYSIS_RUN_DIR is None and not results.empty:
    ANALYSIS_RUN_DIR = Path(results.sort_values("best_val_map_75_95", ascending=False).iloc[0]["run_dir"])
if ANALYSIS_RUN_DIR is not None:
    ANALYSIS_RUN_DIR = Path(ANALYSIS_RUN_DIR)
    validation_payload = predict_run(ANALYSIS_RUN_DIR, val_dataset, bundle, CONFIG, with_targets=True)
    ANALYSIS_DIR = ANALYSIS_RUN_DIR / "analysis"
    ANALYSIS_DIR.mkdir(exist_ok=True)
    print("분석 대상:", ANALYSIS_RUN_DIR)
    print("추론 기록:", validation_payload["speed"])
else:
    print("분석할 완료 실행이 없습니다. 후보 하나를 학습한 뒤 다시 실행하세요.")

### 셀 14 · 전체·클래스별·IoU별 AP


In [ ]:
report = None
if validation_payload is not None:
    report = evaluate_records(validation_payload["records"], bundle["class_map"], detailed=True)
    display(pd.DataFrame([report["summary"]]))
    display(report["classes"].sort_values("ap_75_95", na_position="last"))
    display(report["ious"])
    write_json(ANALYSIS_DIR / "summary.json", report["summary"])
    report["classes"].to_csv(ANALYSIS_DIR / "class_ap.csv", index=False)
    report["ious"].to_csv(ANALYSIS_DIR / "iou_ap.csv", index=False)
    np.savez_compressed(ANALYSIS_DIR / "precision_recall.npz", precision=report["precision"],
                        category_ids=report["category_ids"])
else:
    print("셀 13에서 validation 예측을 먼저 준비하세요.")

### 셀 15 · 학습 곡선·AP 그래프


In [ ]:
if report is not None:
    for filename, figure in [
        ("learning_curves.png", plot_learning_curves([ANALYSIS_RUN_DIR])),
        ("class_iou_ap.png", plot_ap(report)),
    ]:
        figure.savefig(ANALYSIS_DIR / filename, dpi=150, bbox_inches="tight")
        plt.show()
        plt.close(figure)

혼동행렬은 **행=정답, 열=예측**, 마지막 행은 추가 예측, 마지막 열은 미검출이다.
IoU 기반 일대일 대응은 약 종류가 다른 예측도 연결한다. 아래 score 기준은 진단·표시용이며 mAP용 예측을 자르지 않는다.
`localization_or_missed`는 낮은 IoU의 같은 클래스 예측이 있다는 진단 후보다. 원인 확정은 이미지로 확인한다.
이미지의 `I:`는 해당 예측과 모든 정답 중 최대 IoU다. 정식 mAP의 클래스별 대응 결과와 구분한다.

### 셀 16 · 검출 혼동행렬·오류 이미지


In [ ]:
DIAGNOSTIC_IOU = 0.75
DIAGNOSTIC_SCORE = 0.30
if validation_payload is not None:
    matrix, errors = detection_errors(validation_payload["records"], bundle["class_map"]["category_id"],
                                       iou_threshold=DIAGNOSTIC_IOU, score_threshold=DIAGNOSTIC_SCORE)
    matrix.to_csv(ANALYSIS_DIR / "confusion_counts.csv")
    errors.to_csv(ANALYSIS_DIR / "error_events.csv", index=False)
    write_json(ANALYSIS_DIR / "diagnostic_settings.json", {"iou": DIAGNOSTIC_IOU, "score": DIAGNOSTIC_SCORE,
                                                          "rows": "true", "columns": "predicted"})
    display(errors["kind"].value_counts().rename_axis("kind").to_frame("events"))
    display(errors.loc[errors["kind"].ne("correct")].head(20))
    for filename, figure in [
        ("confusion_normalized.png", plot_confusion(matrix, normalize=True)),
        ("error_examples.png", preview_records(validation_payload["records"], errors, count=6,
                                               score_threshold=DIAGNOSTIC_SCORE)),
    ]:
        figure.savefig(ANALYSIS_DIR / filename, dpi=150, bbox_inches="tight")
        plt.show()
        plt.close(figure)

### 셀 17 · 설정·seed 비교와 최종 설정 선택


In [ ]:
results = completed_results(OUTPUT_ROOT, CONFIG, bundle)
FINAL_CANDIDATE_ID = None  # 직접 지정하거나, 세 seed가 완료된 후보의 평균 mAP로 선택한다.
FINAL_RUNS = pd.DataFrame()
if not results.empty:
    if results["spec_id"].duplicated().any():
        raise ValueError("동일 설정·seed의 결과가 중복됩니다. 사용할 run만 선택한 뒤 비교하세요.")
    comparison = results.groupby("candidate_id").agg(
        seed_count=("seed", "nunique"), map_mean=("best_val_map_75_95", "mean"),
        map_std=("best_val_map_75_95", "std"), training_seconds=("training_seconds", "mean"),
        inference_ms=("inference_ms", "mean"), peak_gpu_gb=("peak_gpu_gb", "max"),
    ).sort_values("map_mean", ascending=False)
    display(comparison)
    comparison.to_csv(RUN_GROUP_DIR / "candidate_comparison.csv")
    figure = plot_candidate_scores(results)
    figure.savefig(RUN_GROUP_DIR / "candidate_seed_scores.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(figure)
    selection_path = RUN_GROUP_DIR / "selection.json"
    allowed = json.loads(selection_path.read_text())["candidate_ids"] if selection_path.exists() else []
    finalists = comparison.loc[comparison.index.isin(allowed) & comparison["seed_count"].eq(3)]
    if len(finalists) == 2:
        if FINAL_CANDIDATE_ID is None:
            FINAL_CANDIDATE_ID = finalists.index[0]
        if FINAL_CANDIDATE_ID not in finalists.index:
            raise ValueError("최종 후보는 반복 학습을 완료한 상위 후보에서 선택하세요.")
        FINAL_RUNS = results.loc[results["candidate_id"].eq(FINAL_CANDIDATE_ID)].sort_values("seed")
        write_json(RUN_GROUP_DIR / "final_selection.json", {
            "candidate_id": FINAL_CANDIDATE_ID, "basis": "validation_mean_over_seeds_0_1_2",
            "run_dirs": FINAL_RUNS["run_dir"].tolist(),
        })
        print("최종 설정:", FINAL_CANDIDATE_ID)
        display(FINAL_RUNS)
    else:
        print("상위 후보 두 설정 모두 seed 0·1·2가 완료되면 최종 비교합니다.")
else:
    print("아직 완료된 실행이 없습니다.")

### 셀 17-1 · 두 모델 비교용 validation 자료 저장

최종 설정의 세 seed 예측·클래스 매핑·데이터 해시·환경·평가 정책을 저장한다. 같은 캐시가 있으면 재사용한다.

In [ ]:
# 두 모델 비교 노트북을 사용하기 전에 True로 바꿔 실행한다. 재학습은 하지 않는다.
EXPORT_COMPARISON = False
if EXPORT_COMPARISON:
    if FINAL_RUNS.empty:
        raise RuntimeError("최종 설정의 seed 0·1·2가 필요합니다. 셀 17을 먼저 완료하세요.")
    comparison_exports = [export_comparison_run(directory, val_dataset, bundle, CONFIG)
                          for directory in FINAL_RUNS["run_dir"]]
    display(pd.DataFrame(comparison_exports))
else:
    print("비교 자료 미저장: 두 모델 비교 전 EXPORT_COMPARISON=True로 실행하세요.")

## 5. Test 예측·제출

### 셀 18 · validation의 실제 CSV 저장·복원 검증


In [ ]:
if validation_payload is not None:
    roundtrip = verify_roundtrip(validation_payload["records"], bundle["class_map"]["category_id"],
                                 ANALYSIS_DIR / "validation_roundtrip.csv")
    display(pd.DataFrame([roundtrip]))
else:
    print("validation 예측이 없습니다. 셀 13을 먼저 실행하세요.")

### 셀 19 · 제출할 실행 선택·Test 전체 추론


In [ ]:
# 빈 목록이면 셀 17에서 확정한 최종 설정의 세 seed를 사용한다.
# 후보 하나로 연결만 확인하려면 [str(ANALYSIS_RUN_DIR)]를 직접 지정할 수 있다.
SUBMISSION_RUN_DIRS = []
TEST_IMAGE_DIR = DATA_ROOT / "raw/test_images"
if not SUBMISSION_RUN_DIRS and not FINAL_RUNS.empty:
    SUBMISSION_RUN_DIRS = FINAL_RUNS["run_dir"].tolist()
test_payloads = {}
if SUBMISSION_RUN_DIRS:
    test_images = test_dataset(TEST_IMAGE_DIR)
    if CACHE_ROOT is not None:
        test_images = cache_dataset(test_images, CACHE_ROOT)
    SUBMISSION_ROOT.mkdir(parents=True, exist_ok=True)
    test_images.frame.to_csv(SUBMISSION_ROOT / "test_manifest.csv", index=False)
    for directory in SUBMISSION_RUN_DIRS:
        directory = Path(directory)
        # 실제 제출할 각 실행에서도 validation CSV 변환을 확인한다.
        val_check = predict_run(directory, val_dataset, bundle, CONFIG, with_targets=True)
        verify_roundtrip(val_check["records"], bundle["class_map"]["category_id"],
                         directory / "analysis/validation_roundtrip.csv")
        payload = predict_run(directory, test_images, bundle, CONFIG, with_targets=False)
        if len(payload["records"]) != len(test_images):
            raise RuntimeError("Test 전체 추론이 완료되지 않았습니다. 해당 run의 추론을 다시 실행하세요.")
        test_payloads[str(directory)] = payload
        print(directory.name, payload["speed"])
        figure = preview_records(payload["records"], count=3, score_threshold=0.30)
        figure.savefig(directory / "test_preview.png", dpi=150, bbox_inches="tight")
        plt.show()
        plt.close(figure)
else:
    print("제출할 실행을 정한 뒤 SUBMISSION_RUN_DIRS를 지정하거나 셀 17을 완료하세요.")

### 셀 20 · CSV 생성·검증·제출 점수 기록


In [ ]:
# Kaggle에 직접 제출한 뒤 run 폴더명별 점수를 입력한다. 자동 제출하지 않는다.
KAGGLE_PUBLIC_SCORES = {}
exports = []
for directory, payload in test_payloads.items():
    directory = Path(directory)
    output_dir = SUBMISSION_ROOT / RUN_GROUP_DIR.name
    output_dir.mkdir(parents=True, exist_ok=True)
    csv_path = output_dir / f"{directory.name}.csv"
    frame = submission_frame(payload["records"], bundle["class_map"]["category_id"])
    if frame.empty:
        raise RuntimeError("제출 예측이 비어 있습니다. 모델·score·Test 이미지를 확인하세요.")
    frame.to_csv(csv_path, index=False)
    loaded = pd.read_csv(csv_path)
    restored = restore_submission(loaded, payload["records"])
    submission_frame(restored, bundle["class_map"]["category_id"])
    if len(loaded) != len(frame):
        raise RuntimeError("저장한 CSV의 행 수가 다릅니다. 파일을 다시 생성하세요.")
    exports.append({"run": directory.name, "csv_path": str(csv_path),
                    "test_images": len(payload["records"]), "prediction_rows": len(frame)})
    score_path = directory / "kaggle_score.json"
    score = KAGGLE_PUBLIC_SCORES.get(directory.name)
    if score is not None:
        if not np.isfinite(score) or not 0 <= score <= 1:
            raise ValueError("Kaggle 점수는 0~1의 유한한 값으로 입력하세요.")
        write_json(score_path, {"public_score": float(score), "csv_path": str(csv_path)})
if exports:
    display(pd.DataFrame(exports))
    pd.DataFrame(exports).to_csv(output_dir / "submission_manifest.csv", index=False)
    print("Kaggle에는 csv_path의 예측 CSV만 제출합니다. manifest는 제출하지 않습니다.")
else:
    print("Test 예측을 먼저 준비하세요. 셀 19의 실행 선택을 확인하세요.")